# V14 — Readout **orde-dua (Gram/bilinear)** pada DINOv3 ViT-7B beku

**Status:** notebook siap jalan, belum pernah dieksekusi siapa pun. Semua angka di bawah adalah angka historis dari `results/`, bukan hasil run ini.

---

## 1. Kenapa arah ini, setelah V13

V13 baru saja menutup hipotesis kernel resize — dan buktinya lebih tajam daripada sekadar skor: preflight mengukur **cosine similarity fitur bicubic vs bilinear = 0,99941**. Piksel jelas berbeda (max abs delta 0,84) tetapi representasi backbone nyaris tidak bergerak (max abs feature delta 0,042). Backbone praktis invarian terhadap pilihan kernel pada geometri ini, jadi seluruh keluarga kernel mati sekaligus.

Yang tersisa: **cara kita membaca token, bukan cara kita menyiapkan piksel.**

Dan di situ ada lubang yang belum pernah disentuh sepanjang V1–V13. Readout champion adalah

```
final CLS  ‖  mean(256 patch token)
```

`mean` adalah statistik **orde-satu**. Ia membuang seluruh struktur **ko-okurensi antar-dimensi fitur** — persis kuantitas yang dalam literatur mewakili *tekstur*. Dua permukaan bisa punya rata-rata token yang hampir sama tetapi pola ko-okurensi yang sangat berbeda: kertas kusut vs daun kering, kilap plastik vs kulit buah. Itu adalah batas R↔O, yaitu 242 dari 287 error (84,3%).

**Penting — ini bukan pengulangan V10-rich.** V10-rich menambah 15 cabang yang semuanya tetap **orde-satu**: `std`, `max`, mean top-k salient, mean per kuadran. Semuanya statistik *marginal* per dimensi atau rata-rata subset. Tidak satu pun menangkap **off-diagonal** ko-okurensi antar-dimensi. Itulah yang ditambahkan V14, dan itu kategori informasi yang berbeda, bukan variasi.

## 2. Dasar dari literatur

Tiga sumber independen menunjuk ke objek yang sama:

1. **DINOv3 sendiri (`ref/papers/2508.10104v1.md` §4, "Gram Anchoring").** Kontribusi utama DINOv3 adalah fase refinement yang meregularisasi **matriks Gram — matriks seluruh dot product berpasangan antar patch feature dalam satu citra**, dihitung pada fitur lokal yang **sudah di-L2-normalisasi**. Artinya struktur Gram adalah kuantitas yang backbone ini secara eksplisit *dilatih untuk menjaga kualitasnya*. Membaca lewat Gram berarti membaca lewat sesuatu yang dijamin oleh objective pretraining-nya sendiri.
2. **FSSDINO (`ref/papers/2602.07550v1.md`, eq. 5–6).** Membangun Gram per kelas `Gc = (1/Nc) Σ f̃ f̃ᵀ` dari fitur ter-L2-normalisasi, lalu memakainya untuk memproyeksikan fitur query. Tabel V mereka: prototype-only sudah kuat, **Gram-only** juga kompetitif, dan **kombinasi keduanya paling baik** — pola yang persis dipakai V14 (cabang orde-satu + cabang orde-dua, difusikan).
3. **Bilinear CNN (Lin et al. 2015, tertaut di CONTEXT §12).** Metode kanonik untuk *fine-grained recognition*: pooling orde-dua atas peta fitur, dengan dua detail yang wajib dan sering dilupakan — **signed square-root** lalu **L2-normalisasi**. Tanpa keduanya fitur bilinear berskala buruk dan tidak terlatih dengan baik. V14 menerapkan keduanya.

## 3. Apa yang dihitung

Untuk tiap citra, satu forward pass `forward_features` (V13 sudah membuktikan ini **bit-exact** setara jalur V10):

**Cabang A — orde-satu (persis champion, tidak berubah):**
```
CLS ‖ mean(256 true patch token)          -> 8192-D
```

**Cabang B — orde-dua (baru):**
```
S = L2_normalize(256 true patch token, dim=-1)     [256, 4096]   <- formulasi Gram DINOv3
Y = S @ P                                          [256, 256]    <- P: proyeksi acak Gaussian TETAP, seed 42
C = Yᵀ Y / 256                                     [256, 256]    <- Gram ko-okurensi fitur
```
`C` yang di-cache, bukan token-nya. Itu bukan kompromi: untuk setiap head berbentuk "proyeksi linear lalu Gram", `C` adalah **statistik cukup** yang eksak — karena `Wᵀ(YᵀY)W = (YW)ᵀ(YW)`. Jadi head tetap bebas mempelajari proyeksinya sendiri, tanpa perlu menyimpan 55 GB token mentah.

**Head cabang orde-dua (yang dipelajari):**
```
G = Wᵀ C W                    W: [256, 64] dipelajari, init orthogonal
g = triu(G), off-diagonal ×√2      -> 2080-D   (skala menjaga norma Frobenius)
g = sign(g)·√|g|                                  <- signed sqrt, Lin et al.
g = L2_normalize(g)                               <- L2, Lin et al.
g = LayerNorm -> Linear(256) -> GELU -> Dropout(0.1)
```

**Fusi (satu forward graph, satu checkpoint):**
```
concat(8192 orde-satu, 256 orde-dua) = 8448
-> LayerNorm -> Linear(512) -> GELU -> Dropout(0.4) -> Linear(3)
```

Ini sah menurut CONTEXT §4: *"Branch internal dalam satu arsitektur yang di-train bersama boleh, asal menghasilkan satu checkpoint dan satu forward graph."* Tidak ada blending, tidak ada head kedua, tidak ada rata-rata probabilitas.

---

## 4. Tiga arm, dan kenapa arm ketiga ada

| Arm | Isi | Peran |
|---|---|---|
| `control` | Head champion V10 apa adanya pada 8192-D | Kontrol utama. **Wajib mereproduksi 0.9901662323927614 persis** — V13 sudah membuktikan ini bisa bit-exact |
| `gram_texture` | Cabang orde-satu + orde-dua, difusikan | Kandidat |
| `capacity_control` | Arsitektur champion, hidden 640 (bukan 512) | **Diagnostik.** Menjawab "apakah gain-nya dari struktur orde-dua, atau cuma dari parameter lebih banyak?" |

Tanpa `capacity_control`, kemenangan `gram_texture` tidak dapat ditafsirkan — kandidat itu memang punya parameter lebih banyak. Kalau `capacity_control` juga menang dengan besaran serupa, kesimpulannya **kapasitas, bukan orde-dua**, dan kandidat harus ditolak walaupun angkanya naik.

Ketiganya dilatih **dari nol** dengan seed, fold, optimizer, jadwal, dan anggaran epoch yang identik. Ini sekaligus menutup confound yang dirancang untuk didiagnosis oleh `V11A1_HEAD_ONLY_CAUSAL_MATERIAL_CONFIRMATION.ipynb` (kandidat warm-start dapat optimisasi ekstra dibanding kontrol) — di sini confound itu tidak bisa muncul sama sekali karena tidak ada warm-start.

---

## 5. Gerbang adopsi — dikunci sebelum ada satu pun angka

`gram_texture` diadopsi **hanya bila kesembilannya PASS**:

1. Integritas pipeline: `control` mereproduksi `0.9901662323927614` dalam ±0.0010.
2. Gain full 5-fold OOF Macro F1 atas `control` ≥ **+0.0003**.
3. Total error OOF **turun**.
4. Error R/O OOF **turun**.
5. Menang ≥ **3 dari 4** fold penilaian (fold 1–4).
6. Electronic F1 tidak turun > **0.0005**.
7. Gain tidak terkonsentrasi: satu fold ≤ **75%** dari total gain positif.
8. Tanda gain **konsisten** pada seed 42, 314, 2026.
9. Tidak ada akses test selama seleksi.

**Dibaca bersama gate, bukan sebagai gate** (dicetak otomatis):

- **Gain vs derau seed.** V13 mengukur std seed kontrol = **0,000213**. Gate +0,0003 hanya ≈ 1,4 σ. Notebook mencetak `gain > 2 × std` secara eksplisit. Gain di bawah itu jangan diklaim sebagai kemenangan walaupun gate lolos.
- **Perbandingan `capacity_control`.** Kalau kapasitas menjelaskan gain, tolak.
- **Kejenuhan epoch.** Kalau `best_epoch` kandidat menumpuk di 15 (batas), kandidat itu *under-trained* dan hasilnya adalah batas bawah, bukan kesimpulan.

Kalau `gram_texture` ditolak, submission tetap dibuat — dari `control`, yaitu resep V10. Apa pun hasilnya notebook berakhir dengan **satu model, satu forward pass, satu argmax**, satu CSV tervalidasi.

---

## 6. Yang notebook ini sadar tidak akan capai

Target >0.997 menuntut error turun 287 → ±60–100. Perlu dinyatakan terus terang: **perubahan readout pada backbone beku hampir pasti tidak akan mencapai itu**, dan ada alasan struktural di CONTEXT §15.5 — karena validasi selalu memakai label resmi asli, setiap perbaikan akurasi *sebenarnya* yang melewati sampel bermasalah justru menurunkan OOF terukur. OOF punya plafon label.

Yang dikejar notebook ini realistis: apakah informasi orde-dua memberi **pengurangan error R/O yang nyata dan stabil lintas seed**. Kalau ya, itu champion baru dan sekaligus arah untuk kompetisi berikutnya. Kalau tidak, itu hasil negatif yang bersih dan menutup satu lagi cabang besar.

---

## 7. Anggaran runtime (T4 × 2, diukur dari V13)

| Tahap | Perkiraan |
|---|---|
| Unduh + dispatch checkpoint | 15–30 menit |
| Ekstraksi train (satu pass, dua cabang sekaligus) | **± 1,4 jam** (V13 terukur 1,38 j/varian) |
| 3 arm × 5 fold × 3 seed head | ± 20 menit |
| Diagnostik grouped-CV | ± 3 menit |
| Ekstraksi test + inferensi + submission | ± 8 menit |
| **Total** | **± 2,3 jam** |

Cache: orde-satu 0,87 GB + Gram 3,47 GB ≈ **4,3 GB**. Resumable dan atomik.

## 8. Instruksi runtime — baca sebelum `Run All`

1. **Accelerator: `GPU T4 x2`.** Notebook menolak konfigurasi lain.
2. **Internet: ON**, dan **Kaggle Secret `HF_TOKEN`** tersedia dengan lisensi DINOv3 sudah disetujui.
3. **Dataset resmi terpasang**: satu folder `train` (9.999 / 3.961 / 12.567), satu folder `test` (1.458), satu `submission.csv` resmi.
4. **Jangan lampirkan CSV berlabel-manual test.** Ada penjaga yang menghentikan notebook.
5. **Isi `TEAM_NAME`.** Pada run V13 nilainya tertinggal `CHANGE_ME`, sehingga berkasnya keluar sebagai `submission_CHANGE_ME.csv`. Kali ini isi dulu.
6. **Mulai dari sesi bersih** supaya kedua GPU kosong saat checkpoint dimuat.
7. Sesudah selesai, sel terakhir mencetak **V14 RESULT BLOCK**. Salin blok itu apa adanya.

> Cache Gram berukuran ± 3,47 GB. Kalau `/kaggle/working` sempit, turunkan `GRAM_PROJECTION_DIM` dari 256 ke 128 (cache jadi ± 0,87 GB). Itu satu-satunya knob yang aman diubah tanpa merusak perbandingan, karena kedua arm tetap memakai nilai yang sama.

In [ ]:
import os

# Set before torch is imported. Fragmentation guard only, not a correctness fix.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import gc
import hashlib
import json
import math
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from torchvision.transforms.functional import InterpolationMode
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# FILL THIS IN BEFORE SUBMITTING (V13 shipped as submission_CHANGE_ME.csv)
# ---------------------------------------------------------------------------
TEAM_NAME = 'CHANGE_ME'

EXPERIMENT_NAME = 'v14_dinov3_vit7b_second_order_gram_texture'
REVISION_MARKER = 'v14_gram_texture_v1'
EXPERIMENT_PURPOSE = (
    'add a second-order (Gram / bilinear) texture branch to the frozen DINOv3 ViT-7B readout; '
    'one change against the V10 champion recipe, all arms trained from scratch in the same session'
)

# ---------------------------------------------------------------------------
# BACKBONE CONTRACT — locked, identical to V10/V13. Not an open question.
# ---------------------------------------------------------------------------
TIMM_MODEL_NAME = 'vit_7b_patch16_dinov3.lvd1689m'
TIMM_HF_REPO_ID = 'timm/vit_7b_patch16_dinov3.lvd1689m'
TRANSFORMERS_MODEL_ID = 'facebook/dinov3-vit7b16-pretrain-lvd1689m'
MODEL_REVISION_REQUEST = 'main'
V10_MODEL_REVISION = 'b40f6c827cb02b13c7a23a76cf598489bb0f94e5'

IMAGE_SIZE = 256
PATCH_SIZE_EXPECTED = 16
EXPECTED_PATCH_TOKENS = 256
EXPECTED_PREFIX_TOKENS = 5
EXPECTED_REGISTER_TOKENS = 4
EXPECTED_SEQUENCE_TOKENS = 261
EXPECTED_HIDDEN_SIZE = 4096

BACKBONE_FROZEN = True
BACKBONE_DTYPE = torch.float16
BACKBONE_DTYPE_NAME = 'float16'
POOLING_COMPUTE_DTYPE = torch.float32
POOLING_COMPUTE_DTYPE_NAME = 'float32'
FEATURE_STORAGE_DTYPE = torch.float16
FEATURE_STORAGE_DTYPE_NAME = 'float16'

SHARDING_STRATEGY = 'explicit_contiguous_two_gpu_streamed'
GPU_STATIC_MEMORY_LIMIT_GIB = 10.5
GPU_MIN_RUNTIME_HEADROOM_GIB = 3.0
CHECKPOINT_MIN_TOTAL_GIB = 1.0
CHECKPOINT_MAX_TOTAL_GIB = 60.0
OUTPUT_SIDE_MODULE_NAMES = ('norm', 'fc_norm', 'head_norm', 'attn_pool', 'head_drop', 'head', 'head_dist')

FEATURE_BATCH_SIZE = 4
FEATURE_LOADER_WORKERS = 2
FEATURE_SHARD_SIZE = 256
FEATURE_MAX_ABS_LIMIT = 1e6
MIN_EXTRACTION_IMAGES_PER_SECOND = 0.8
BENCHMARK_WARMUP_IMAGES = 8
BENCHMARK_TIMED_IMAGES = 24

# ---------------------------------------------------------------------------
# PREPROCESSING — locked to the champion. V13 proved the resize kernel is a
# no-op for this backbone (feature cosine bicubic vs bilinear = 0.99941), so
# there is nothing left to explore here and it stays fixed.
# ---------------------------------------------------------------------------
INTERPOLATION_NAME = 'bilinear'
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
LARGE_IMAGE_MP_THRESHOLD = 8_000_000
LARGE_IMAGE_MAX_SOURCE_SIDE = 4096
SAFE_DECODE_MAX_SIDE = 1024
PIL_MAX_IMAGE_PIXELS = 200_000_000

# ---------------------------------------------------------------------------
# THE SINGLE EXPERIMENTAL VARIABLE: a second-order readout branch
# ---------------------------------------------------------------------------
FIRST_ORDER_BRANCHES = ('final_cls', 'final_patch_mean')
FIRST_ORDER_DIM = len(FIRST_ORDER_BRANCHES) * EXPECTED_HIDDEN_SIZE   # 8192

# Fixed Gaussian random projection applied to L2-normalised patch tokens before
# the Gram. Johnson-Lindenstrauss: preserves inner products, needs no learning,
# and is generated from a fixed seed so it is identical for train and test.
GRAM_PROJECTION_DIM = 256          # width of the cached Gram matrix
GRAM_PROJECTION_SEED = 42
GRAM_RANK = 64                     # learned rank inside the head
GRAM_TRIU_DIM = GRAM_RANK * (GRAM_RANK + 1) // 2   # 2080
GRAM_FUSION_DIM = 256
GRAM_BRANCH_DROPOUT = 0.10

CONTROL_ARM = 'control'
GRAM_ARM = 'gram_texture'
CAPACITY_ARM = 'capacity_control'
ARM_ORDER = (CONTROL_ARM, GRAM_ARM, CAPACITY_ARM)
DIAGNOSTIC_ARMS = (CAPACITY_ARM,)   # never eligible for adoption

# ---------------------------------------------------------------------------
# OFFICIAL DATA CONTRACT
# ---------------------------------------------------------------------------
CLASS_DIR_TO_LABEL = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_RAW_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_TRAIN_ROWS = 26527
EXPECTED_TEST_ROWS = 1458
SUBMISSION_COLUMNS = ['id', 'predicted']
ALLOWED_IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# ---------------------------------------------------------------------------
# HEAD PROTOCOL — identical for every arm
# ---------------------------------------------------------------------------
SEED = 42
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
ASSESSMENT_FOLDS = (1, 2, 3, 4)
STABILITY_SEEDS = (314, 2026)
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
HEAD_LEARNING_RATE = 8e-4
HEAD_WEIGHT_DECAY = 1e-4
HEAD_BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 512
GRAD_CLIP_MAX_NORM = 1.0
LABEL_SMOOTHING = 0.05
HEAD_HIDDEN_DIM = 512
HEAD_DROPOUT = 0.40
CAPACITY_HIDDEN_DIM = 640          # capacity_control only

# ---------------------------------------------------------------------------
# LOCKED ADOPTION GATES (fixed before any V14 result exists)
# ---------------------------------------------------------------------------
ADOPTION_MIN_OOF_MACRO_F1_GAIN = 0.0003
ADOPTION_MIN_ASSESSMENT_FOLDS_WON = 3
ADOPTION_MAX_ELECTRONIC_F1_DROP = 0.0005
ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE = 0.75
STRONG_EVIDENCE_MACRO_F1_GAIN = 0.0015
CONTROL_REPRODUCTION_TOLERANCE = 0.0010
RELOAD_TOLERANCE = 1e-5

# Grouped-CV diagnostic (never a gate)
GROUP_SIMILARITY_THRESHOLDS = (0.95, 0.98, 0.99)
V9_REFERENCE_GROUPED_FILES = 910
V9_REFERENCE_GROUP_COUNT = 436

# ---------------------------------------------------------------------------
# MEASURED HISTORICAL FACTS (from results/, never a claim about this run)
# ---------------------------------------------------------------------------
V10_REFERENCE = {
    'source': 'results/v10_dinov3_vit7b_timm_rich_spatial + results/v13_.../selection_decision.json',
    'fold_assignment_sha256': '0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c',
    'labels_sha256': '719d36e9918b1fd09711c06d1b259fca08ca6eb4d973de1388d5e3ddecf22d35',
    'ordered_image_ids_sha256': '210f8f14797be49bad962e81f1da660a257ec28ee2136b54a7516950a66cb2e8',
    'oof_macro_f1': 0.9901662323927614,
    'oof_total_errors': 287,
    'oof_ro_errors': 242,
    'oof_electronic_errors': 16,
    'oof_f1_electronic': 0.994328922495274,
    'oof_recyclable_to_organic': 96,
    'oof_organic_to_recyclable': 146,
    'oof_high_confidence_errors': 33,
    'per_fold_macro_f1': {
        '0': 0.9897706675032462, '1': 0.9903469894805695, '2': 0.9908291600511604,
        '3': 0.9890345290445208, '4': 0.9908498966736979,
    },
    'best_epochs': {'0': 5, '1': 8, '2': 3, '3': 5, '4': 6},
    'head_parameter_count': 4212739,
    'full_training_epochs': 5,
    'test_predicted_class_distribution': {'0': 520, '1': 201, '2': 737},
    'v13_reproduced_it_exactly': True,
}

# Measured in V13 across seeds 42 / 314 / 2026 on the control arm.
V13_CONTROL_SEED_SPREAD = 0.00021296796880071838
V13_CONTROL_SEED_MACRO_F1 = {'42': 0.9901662323927614, '314': 0.9898990773387165, '2026': 0.9896446210609852}
V13_CONTROL_SEED_TOTAL_ERRORS = {'42': 287, '314': 295, '2026': 308}
V13_NOTE = (
    'V13 rejected the bicubic resize kernel: feature cosine vs bilinear was 0.99941, so the backbone is '
    'effectively invariant to the resize kernel. The whole kernel family is closed, not just bicubic.'
)

# ---------------------------------------------------------------------------
# CONTRACT ASSERTIONS
# ---------------------------------------------------------------------------
assert IMAGE_SIZE == 256 and PATCH_SIZE_EXPECTED == 16
assert EXPECTED_SEQUENCE_TOKENS == EXPECTED_PREFIX_TOKENS + EXPECTED_PATCH_TOKENS
assert EXPECTED_PREFIX_TOKENS == EXPECTED_REGISTER_TOKENS + 1
assert BACKBONE_FROZEN and BACKBONE_DTYPE is torch.float16
assert POOLING_COMPUTE_DTYPE is torch.float32 and FEATURE_STORAGE_DTYPE is torch.float16
assert SEED == 42 and N_FOLDS == 5 and DEVELOPMENT_FOLD == 0 and ASSESSMENT_FOLDS == (1, 2, 3, 4)
assert sum(EXPECTED_RAW_COUNTS.values()) == EXPECTED_TRAIN_ROWS
assert ARM_ORDER[0] == CONTROL_ARM, 'the control must run first so it can validate the pipeline'
assert GRAM_ARM not in DIAGNOSTIC_ARMS, 'the candidate must be adoption-eligible'
assert CAPACITY_ARM in DIAGNOSTIC_ARMS, 'the capacity arm must never be adoptable'
assert 1 <= GRAM_RANK <= GRAM_PROJECTION_DIM <= EXPECTED_HIDDEN_SIZE
assert GRAM_TRIU_DIM == GRAM_RANK * (GRAM_RANK + 1) // 2
assert SEED not in STABILITY_SEEDS
assert INTERPOLATION_NAME == 'bilinear', 'V13 closed the kernel question; do not reopen it here'

TEST_ACCESS_STATE = {'test_images_accessed': False, 'manual_test_labels_accessed': False}
NOTEBOOK_STARTED_AT = time.perf_counter()

print('Experiment :', EXPERIMENT_NAME)
print('Arms       :')
for arm in ARM_ORDER:
    role = 'diagnostic only' if arm in DIAGNOSTIC_ARMS else ('control' if arm == CONTROL_ARM else 'CANDIDATE')
    print('  {:<18} {}'.format(arm, role))
print('Second-order: patch tokens L2-normalised -> fixed random proj {} -> Gram {}x{} cached'.format(
    GRAM_PROJECTION_DIM, GRAM_PROJECTION_DIM, GRAM_PROJECTION_DIM
))
print('              head learns {} -> {} then triu {} dims, signed-sqrt + L2 (Lin et al. 2015)'.format(
    GRAM_PROJECTION_DIM, GRAM_RANK, GRAM_TRIU_DIM
))
print('Gram cache  : {:.2f} GiB'.format(
    EXPECTED_TRAIN_ROWS * GRAM_PROJECTION_DIM * GRAM_PROJECTION_DIM * 2 / 2 ** 30
))
print('Adoption    : OOF gain >= {:+.4f}; seed noise measured in V13 = {:.6f} (2 sigma = {:.6f})'.format(
    ADOPTION_MIN_OOF_MACRO_F1_GAIN, V13_CONTROL_SEED_SPREAD, 2 * V13_CONTROL_SEED_SPREAD
))
if TEAM_NAME == 'CHANGE_ME':
    print()
    print('!' * 78)
    print('!! TEAM_NAME is still "CHANGE_ME" -- V13 already shipped that way. Set it now.  !!')
    print('!' * 78)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')
Image.MAX_IMAGE_PIXELS = PIL_MAX_IMAGE_PIXELS
ImageFile.LOAD_TRUNCATED_IMAGES = False

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError('This notebook requires a Kaggle runtime with /kaggle/working and GPU T4 x2.')
KAGGLE_INPUT = Path('/kaggle/input')
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
CACHE_DIR = ARTIFACT_DIR / 'feature_cache'
HEADS_DIR = ARTIFACT_DIR / 'heads'
for directory in (ARTIFACT_DIR, CACHE_DIR, HEADS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SELECTION_DECISION_PATH = ARTIFACT_DIR / 'selection_decision.json'
PREFLIGHT_REPORT_PATH = ARTIFACT_DIR / 'preflight_report.json'


def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError('Not JSON serializable: {}'.format(type(value).__name__))


def sha256_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, default=json_default)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def write_npy_atomic(array, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp.npy')
    np.save(temporary, array)
    os.replace(temporary, path)


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def host_rss_gib():
    resident_pages = int(Path('/proc/self/statm').read_text().split()[1])
    return resident_pages * os.sysconf('SC_PAGE_SIZE') / 2 ** 30


def gpu_memory_rows():
    return [
        {
            'device_id': device_id,
            'allocated_gib': torch.cuda.memory_allocated(device_id) / 2 ** 30,
            'reserved_gib': torch.cuda.memory_reserved(device_id) / 2 ** 30,
            'peak_allocated_gib': torch.cuda.max_memory_allocated(device_id) / 2 ** 30,
        }
        for device_id in range(torch.cuda.device_count())
    ]


def cuda_memory_snapshot(label):
    rows = []
    for device_id in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        rows.append({
            'label': label,
            'device_id': device_id,
            'free_gib': free_bytes / 2 ** 30,
            'total_gib': total_bytes / 2 ** 30,
            'allocated_gib': torch.cuda.memory_allocated(device_id) / 2 ** 30,
            'reserved_gib': torch.cuda.memory_reserved(device_id) / 2 ** 30,
        })
    return rows


def release_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sanitize_for_filename(value):
    cleaned = ''.join(character if character.isalnum() or character in {'-', '_'} else '_' for character in str(value))
    return cleaned.strip('_') or 'UNSET'


def elapsed_minutes():
    return (time.perf_counter() - NOTEBOOK_STARTED_AT) / 60.0


print('Artifact directory:', ARTIFACT_DIR)
print('Seed locked at    :', SEED)

In [ ]:
import accelerate
import safetensors
import timm
import timm.data  # explicit: timm/__init__.py does not import the data subpackage

ENVIRONMENT_INFO = {
    'python_version': sys.version,
    'platform': platform.platform(),
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
    'cudnn_version': torch.backends.cudnn.version(),
    'timm_version': timm.__version__,
    'accelerate_version': accelerate.__version__,
    'safetensors_version': safetensors.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
}

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError(
        'V14 requires exactly two visible CUDA devices, found {}. Set the accelerator to "GPU T4 x2".'.format(
            torch.cuda.device_count() if torch.cuda.is_available() else 0
        )
    )
GPU_RUNTIME_INFO = []
for device_id in (0, 1):
    properties = torch.cuda.get_device_properties(device_id)
    upper_name = properties.name.upper()
    if 'TESLA T4' not in upper_name and upper_name.strip() != 'T4':
        raise RuntimeError('cuda:{} is "{}", not a Tesla T4.'.format(device_id, properties.name))
    GPU_RUNTIME_INFO.append({
        'device_id': device_id,
        'name': properties.name,
        'total_vram_gib': properties.total_memory / 2 ** 30,
        'multi_processor_count': properties.multi_processor_count,
    })

if not KAGGLE_INPUT.is_dir():
    raise RuntimeError('Attach the competition dataset: /kaggle/input was not found.')

blocked_label_files = []
for candidate_path in KAGGLE_INPUT.rglob('*.csv'):
    normalized = candidate_path.name.lower().replace('_', '-')
    if 'manual' in normalized and 'label' in normalized:
        blocked_label_files.append(str(candidate_path))
if blocked_label_files:
    raise RuntimeError(
        'Competition safety guard: manual test-label CSV files are attached as notebook inputs. Detach them:'
        + '\n' + '\n'.join(blocked_label_files)
    )

LOCKED_CONTRACT = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'experiment_purpose': EXPERIMENT_PURPOSE,
    'backbone_backend': 'timm',
    'timm_model_name': TIMM_MODEL_NAME,
    'timm_hf_repo_id': TIMM_HF_REPO_ID,
    'transformers_model_id': TRANSFORMERS_MODEL_ID,
    'backbone_frozen': BACKBONE_FROZEN,
    'backbone_dtype': BACKBONE_DTYPE_NAME,
    'pooling_compute_dtype': POOLING_COMPUTE_DTYPE_NAME,
    'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
    'image_size': IMAGE_SIZE,
    'interpolation': INTERPOLATION_NAME,
    'geometry': 'direct_square 256x256, antialias True, no center crop, no letterbox, no augmentation',
    'token_contract': {
        'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
        'prefix_token_count': EXPECTED_PREFIX_TOKENS,
        'register_token_count': EXPECTED_REGISTER_TOKENS,
        'patch_token_count': EXPECTED_PATCH_TOKENS,
        'hidden_size': EXPECTED_HIDDEN_SIZE,
        'cls_position': 'prefix token 0',
        'register_tokens_excluded_from_every_patch_statistic': True,
    },
    'first_order_readout': {
        'branches': list(FIRST_ORDER_BRANCHES),
        'dim': FIRST_ORDER_DIM,
        'definition': 'final normalised CLS concatenated with the mean of the 256 true patch tokens',
        'identical_to_v10_champion': True,
    },
    'second_order_readout': {
        'patch_token_normalisation': 'L2 per token, matching the DINOv3 Gram-anchoring formulation',
        'fixed_projection': 'Gaussian random, seed {}, scaled 1/sqrt(d)'.format(GRAM_PROJECTION_SEED),
        'projection_dim': GRAM_PROJECTION_DIM,
        'cached_object': 'C = Y^T Y / num_patches, symmetric [{}, {}]'.format(
            GRAM_PROJECTION_DIM, GRAM_PROJECTION_DIM
        ),
        'sufficiency_note': (
            'C is an exact sufficient statistic for any head of the form "linear projection then Gram", '
            'because W^T (Y^T Y) W = (Y W)^T (Y W). Caching C instead of tokens loses nothing for this head family.'
        ),
        'learned_rank': GRAM_RANK,
        'triu_dim': GRAM_TRIU_DIM,
        'normalisation': 'off-diagonal x sqrt(2), then signed square root, then L2 (Lin et al. 2015)',
        'literature': [
            'DINOv3 2508.10104 sec.4 Gram anchoring: pairwise dot products of L2-normalised patch features',
            'FSSDINO 2602.07550 eq.5-6: class Gram of L2-normalised features; prototype+Gram beats either alone',
            'Bilinear CNN (Lin et al. 2015): second-order pooling with signed sqrt then L2 for fine-grained tasks',
        ],
    },
    'arms': {
        'control': 'V10 champion head on the 8192-D first-order readout, adoption reference',
        'gram_texture': 'first-order plus second-order branch fused into one classifier, THE CANDIDATE',
        'capacity_control': 'champion architecture with hidden {}, DIAGNOSTIC ONLY, never adoptable'.format(
            CAPACITY_HIDDEN_DIM
        ),
    },
    'all_arms_trained_from_scratch': True,
    'warm_start_used': False,
    'sharding_strategy': SHARDING_STRATEGY,
    'seed': SEED,
    'stability_seeds': list(STABILITY_SEEDS),
    'n_folds': N_FOLDS,
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'head_protocol': {
        'max_epochs': MAX_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'batch_size': HEAD_BATCH_SIZE,
        'predict_batch_size': PREDICT_BATCH_SIZE,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
        'label_smoothing': LABEL_SMOOTHING,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'class_weights': 'total / (num_classes * class_count) from the current training partition',
        'model_selection_metric': 'validation Macro F1, log loss as secondary diagnostic',
        'threshold_tuning': False,
    },
    'gates': {
        'adoption_min_oof_macro_f1_gain': ADOPTION_MIN_OOF_MACRO_F1_GAIN,
        'adoption_min_assessment_folds_won': ADOPTION_MIN_ASSESSMENT_FOLDS_WON,
        'adoption_max_electronic_f1_drop': ADOPTION_MAX_ELECTRONIC_F1_DROP,
        'adoption_max_single_fold_gain_share': ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE,
        'control_reproduction_tolerance': CONTROL_REPRODUCTION_TOLERANCE,
        'seed_sign_consistency_required': True,
        'measured_seed_noise_from_v13': V13_CONTROL_SEED_SPREAD,
    },
    'single_model_contract': {
        'fold_ensembling': False,
        'seed_ensembling': False,
        'probability_blending': False,
        'model_soup': False,
        'specialist_heads': False,
        'test_time_augmentation': False,
        'multi_crop_inference': False,
        'horizontal_flip_inference': False,
        'multiple_backbone_families': False,
        'manual_prediction_correction': False,
        'note': (
            'the gram_texture arm is two internal branches inside ONE architecture trained jointly, producing '
            'ONE checkpoint and ONE forward graph, which CONTEXT section 4 explicitly permits'
        ),
    },
    'official_raw_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
    'official_test_rows': EXPECTED_TEST_ROWS,
    'historical_reference_v10': V10_REFERENCE,
    'historical_note_v13': V13_NOTE,
}
LOCKED_CONTRACT_SIGNATURE = sha256_json(LOCKED_CONTRACT)

write_json_atomic({'environment': ENVIRONMENT_INFO, 'gpus': GPU_RUNTIME_INFO}, ARTIFACT_DIR / 'environment.json')
write_json_atomic(
    {'locked_contract': LOCKED_CONTRACT, 'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE},
    ARTIFACT_DIR / 'locked_contract.json',
)
write_json_atomic(
    {'team_name': TEAM_NAME, 'artifact_dir': str(ARTIFACT_DIR),
     'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
     'test_access_state_at_configuration_time': dict(TEST_ACCESS_STATE)},
    ARTIFACT_DIR / 'config.json',
)

print(json.dumps(ENVIRONMENT_INFO, indent=2))
for info in GPU_RUNTIME_INFO:
    print('cuda:{device_id}: {name}, {total_vram_gib:.2f} GiB'.format(**info))
print('Manual test-label guard passed.')
print('Locked contract signature:', LOCKED_CONTRACT_SIGNATURE)

In [ ]:
from huggingface_hub import model_info, snapshot_download

HF_ACTION_HINT = (
    'Actions to resolve:' + '\n'
    '  1. Create a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) with a Hugging Face read token.' + '\n'
    '  2. Open https://huggingface.co/{} with that same account and accept the DINOv3 license.' + '\n'
    '  3. Enable notebook internet access.' + '\n'
    '  4. Do not substitute a smaller backbone; the locked protocol requires DINOv3 ViT-7B.'
)


def resolve_hf_token():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            'Missing Hugging Face token for the gated DINOv3 checkpoint.' + '\n'
            + HF_ACTION_HINT.format(TIMM_HF_REPO_ID)
        )
    return token


HF_TOKEN_VALUE = resolve_hf_token()
print('Hugging Face authentication present; the token is never printed or persisted.')

try:
    RESOLVED_REVISION = model_info(TIMM_HF_REPO_ID, revision=MODEL_REVISION_REQUEST, token=HF_TOKEN_VALUE).sha
except Exception as exc:
    raise RuntimeError(
        'Could not resolve an immutable revision for {}.'.format(TIMM_HF_REPO_ID) + '\n'
        + HF_ACTION_HINT.format(TIMM_HF_REPO_ID)
    ) from exc
if not RESOLVED_REVISION or len(RESOLVED_REVISION) != 40 or any(
    character not in '0123456789abcdef' for character in RESOLVED_REVISION.lower()
):
    raise RuntimeError('The Hub did not return an immutable 40-character commit SHA (got {!r}).'.format(
        RESOLVED_REVISION
    ))

MODEL_DOWNLOAD_STARTED_AT = time.perf_counter()
try:
    MODEL_SNAPSHOT_DIR = snapshot_download(
        repo_id=TIMM_HF_REPO_ID,
        revision=RESOLVED_REVISION,
        token=HF_TOKEN_VALUE,
        allow_patterns=['*.safetensors', '*.safetensors.index.json', '*.json'],
    )
except Exception as exc:
    raise RuntimeError(
        'Failed to download {} at revision {}. Check disk space (~25 GB), internet, and license.'.format(
            TIMM_HF_REPO_ID, RESOLVED_REVISION
        ) + '\n' + HF_ACTION_HINT.format(TIMM_HF_REPO_ID)
    ) from exc
MODEL_DOWNLOAD_SECONDS = time.perf_counter() - MODEL_DOWNLOAD_STARTED_AT

MODEL_REVISION_MATCHES_V10 = RESOLVED_REVISION == V10_MODEL_REVISION
snapshot_bytes = sum(path.stat().st_size for path in Path(MODEL_SNAPSHOT_DIR).rglob('*') if path.is_file())
print('Repository        :', TIMM_HF_REPO_ID)
print('Immutable revision:', RESOLVED_REVISION)
print('Matches V10/V13   :', MODEL_REVISION_MATCHES_V10, '(V10/V13 used {})'.format(V10_MODEL_REVISION))
if not MODEL_REVISION_MATCHES_V10:
    print('WARNING: a different revision weakens comparability against the recorded V10 numbers.')
    print('         The control-vs-candidate comparison inside this run stays valid regardless,')
    print('         because every arm reads the same cache from this same revision.')
print('Snapshot size     : {:.2f} GiB (downloaded in {:.1f} s)'.format(
    snapshot_bytes / 2 ** 30, MODEL_DOWNLOAD_SECONDS
))

In [ ]:
def discover_named_dirs(base, wanted, max_depth=3, prune_names=()):
    if not base.is_dir():
        return []
    found = []
    stack = [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            lowered = child.name.lower()
            if lowered in prune_names:
                # Pruned without enumeration so official test images stay untouched during selection.
                continue
            if lowered == wanted:
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found


def exact_raw_train_signature(train_dir):
    if not train_dir.is_dir():
        return None
    try:
        root_entries = list(train_dir.iterdir())
        if {path.name for path in root_entries} != set(CLASS_DIR_TO_LABEL):
            return None
        if any(path.is_symlink() or not path.is_dir() for path in root_entries):
            return None
        counts = {}
        for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
            entries = list((train_dir / class_dir_name).iterdir())
            if any(
                entry.is_symlink() or not entry.is_file() or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
                for entry in entries
            ):
                return None
            counts[label] = len(entries)
        return counts if counts == EXPECTED_RAW_COUNTS else None
    except OSError:
        return None


def resolve_train_dir():
    candidates = [Path.cwd() / 'data' / 'train', KAGGLE_WORKING / 'data' / 'train']
    candidates.extend(discover_named_dirs(KAGGLE_INPUT, 'train', max_depth=3, prune_names={'test'}))
    matches = {
        candidate.resolve() for candidate in candidates
        if exact_raw_train_signature(candidate) == EXPECTED_RAW_COUNTS
    }
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(path) for path in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError(
            'Expected exactly one official raw train directory with counts {}; found {}:'.format(
                EXPECTED_RAW_COUNTS, len(matches)
            ) + '\n' + rendered
        )
    return matches.pop()


def load_rgb_image(path):
    """Identical decode path to V7 / V10 / V13, including the decoder-memory guard."""
    try:
        with Image.open(path) as image:
            source_width, source_height = image.size
            if source_width <= 0 or source_height <= 0:
                raise ValueError('invalid image dimensions')
            needs_downscale = (
                source_width * source_height >= LARGE_IMAGE_MP_THRESHOLD
                or max(source_width, source_height) > LARGE_IMAGE_MAX_SOURCE_SIDE
            )
            image.draft('RGB', (SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE))
            rgb_image = ImageOps.exif_transpose(image).convert('RGB')
            if needs_downscale:
                rgb_image.thumbnail((SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE), Image.Resampling.LANCZOS)
            return rgb_image.copy()
    except Exception as exc:
        raise RuntimeError('Image cannot be decoded: {}'.format(path)) from exc


IMAGE_TRANSFORM = v2.Compose([
    v2.ToImage(),
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

PREPROCESS_CONFIG = {
    'open': 'PIL, EXIF transpose, RGB conversion',
    'large_image_guard': {
        'megapixel_threshold': LARGE_IMAGE_MP_THRESHOLD,
        'max_source_side': LARGE_IMAGE_MAX_SOURCE_SIDE,
        'downscaled_to_max_side': SAFE_DECODE_MAX_SIDE,
        'resample': 'LANCZOS',
        'purpose': 'decoder memory safety only, never a predictive feature',
    },
    'resize_geometry': 'direct_square',
    'spatial_override': {'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    'interpolation': INTERPOLATION_NAME,
    'antialias': True,
    'rescale_factor': 1.0 / 255.0,
    'image_mean': list(IMAGENET_MEAN),
    'image_std': list(IMAGENET_STD),
    'center_crop': False,
    'letterbox': False,
    'augmentation': 'none',
    'feature_view': 'orig',
}
PREPROCESS_SIGNATURE = sha256_json(PREPROCESS_CONFIG)


class ImageRows(Dataset):
    """Deterministic, order-preserving image reader for frozen feature extraction."""

    def __init__(self, paths):
        self.paths = [str(path) for path in paths]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, position):
        return IMAGE_TRANSFORM(load_rgb_image(self.paths[position]))


def index_raw_train(train_dir):
    rows = []
    observed_counts = {}
    for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
        class_dir = train_dir / class_dir_name
        entries = sorted(class_dir.iterdir(), key=lambda path: path.name)
        invalid = [
            entry.name for entry in entries
            if entry.is_symlink() or not entry.is_file() or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
        ]
        if invalid:
            raise RuntimeError('Unsupported entries in {}: {}'.format(class_dir, invalid[:10]))
        observed_counts[label] = len(entries)
        for path in entries:
            rows.append({
                'image_id': path.relative_to(train_dir).as_posix(),
                'label': label,
                'class_name': CLASS_NAMES[label],
                'path': str(path.resolve()),
                'file_size_bytes': path.stat().st_size,
            })
    if observed_counts != EXPECTED_RAW_COUNTS:
        raise RuntimeError('Raw class counts mismatch: observed={}, expected={}.'.format(
            observed_counts, EXPECTED_RAW_COUNTS
        ))
    frame = pd.DataFrame(rows).sort_values('image_id').reset_index(drop=True)
    if len(frame) != EXPECTED_TRAIN_ROWS or not frame['image_id'].is_unique:
        raise RuntimeError('Raw-train row count or image_id uniqueness failed.')
    return frame


TRAIN_DIR = resolve_train_dir()
train_df = index_raw_train(TRAIN_DIR)

# Fold reconstruction: byte-for-byte the V7 / V10 / V13 definition.
train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold_index, (_, valid_positions) in enumerate(splitter.split(train_df, train_df['label'])):
    train_df.loc[valid_positions, 'fold'] = fold_index
train_df['fold'] = train_df['fold'].astype(np.int64)
if set(train_df['fold'].tolist()) != set(range(N_FOLDS)):
    raise RuntimeError('Five-fold assignment is incomplete.')

ORDERED_IMAGE_IDS_SHA256 = sha256_json(train_df['image_id'].tolist())
LABELS_SHA256 = sha256_json(train_df['label'].astype(int).tolist())
FOLD_ASSIGNMENT_SHA256 = sha256_json(train_df[['image_id', 'label', 'fold']].to_dict('records'))
RAW_DATA_SHA256 = sha256_json(train_df[['image_id', 'label', 'file_size_bytes']].to_dict('records'))

# HARD PROTOCOL-MATCH GATE against V10 / V13.
PROTOCOL_MATCH_CHECKS = {
    'ordered_image_ids_match': ORDERED_IMAGE_IDS_SHA256 == V10_REFERENCE['ordered_image_ids_sha256'],
    'labels_match': LABELS_SHA256 == V10_REFERENCE['labels_sha256'],
    'fold_assignment_matches': FOLD_ASSIGNMENT_SHA256 == V10_REFERENCE['fold_assignment_sha256'],
}
failed_protocol_checks = [name for name, passed in PROTOCOL_MATCH_CHECKS.items() if not passed]
if failed_protocol_checks:
    raise RuntimeError(
        'Protocol match against V10/V13 failed: {}. Expected fold sha256 {}, got {}.'.format(
            failed_protocol_checks, V10_REFERENCE['fold_assignment_sha256'], FOLD_ASSIGNMENT_SHA256
        )
    )

FOLD_SIGNATURE = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'fold_definition': {
        'implementation': 'sklearn.model_selection.StratifiedKFold',
        'n_splits': N_FOLDS, 'shuffle': True, 'random_state': SEED,
        'ordering': 'relative official image_id ascending before the split',
    },
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
    'validation_labels': 'always the official original labels; no drop and no relabel',
    'matches_v10_fold_signature': True,
}
DATA_CONTRACT = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'canonical_data': 'official raw labeled train images only',
    'raw_train_root': str(TRAIN_DIR),
    'row_count': int(len(train_df)),
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SHA256,
    'labels_sha256': LABELS_SHA256,
    'raw_data_sha256': RAW_DATA_SHA256,
    'fold_signature': FOLD_SIGNATURE,
    'preprocess_config': PREPROCESS_CONFIG,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'external_labeled_images_used': False,
    'metadata_used_as_feature': False,
    'dropped_or_relabeled_rows': 0,
    'teammate_embedding_cache_used': False,
    'teammate_embedding_cache_note': (
        'research/timm-embedding-satdat is not used. V14 needs a per-image SECOND-ORDER statistic that no '
        'existing cache contains (V7/V10 cache is [N,2,4096] already pooled; V11 cache is [N,64,512] already '
        'projected), so a fresh extraction is required regardless of that cache protocol and mounting it '
        'would save nothing.'
    ),
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
DATA_CONTRACT_SIGNATURE = sha256_json(DATA_CONTRACT)

write_json_atomic(FOLD_SIGNATURE, ARTIFACT_DIR / 'fold_signature.json')
write_json_atomic(
    {'data_contract': DATA_CONTRACT, 'data_contract_signature': DATA_CONTRACT_SIGNATURE},
    ARTIFACT_DIR / 'data_contract.json',
)
write_csv_atomic(train_df[['image_id', 'label', 'class_name', 'fold']], ARTIFACT_DIR / 'fold_assignments.csv')

cached_labels = torch.tensor(train_df['label'].to_numpy(), dtype=torch.long)
cached_folds = torch.tensor(train_df['fold'].to_numpy(), dtype=torch.long)

print('Official raw train root :', TRAIN_DIR)
print('Rows                    :', len(train_df))
print('Class counts            :', train_df['label'].value_counts().sort_index().to_dict())
print('Fold assignment sha256  :', FOLD_ASSIGNMENT_SHA256)
print('  -> matches V10/V13    : True (hard gate passed)')
print('Preprocess signature    :', PREPROCESS_SIGNATURE)
print('Data contract signature :', DATA_CONTRACT_SIGNATURE)

In [ ]:
from accelerate import dispatch_model, init_empty_weights
from accelerate.utils import set_module_tensor_to_device
from safetensors import safe_open

# This whole cell is the V13 loader, unchanged. It ran end to end on Kaggle and reproduced
# the champion bit-exactly, so it is deliberately not re-derived.

DISPATCH_FAILURE_HINT = (
    'The DINOv3 ViT-7B backbone could not be placed across the two Tesla T4 GPUs.' + '\n'
    'This is a weight-loading failure before any image is processed, so changing FEATURE_BATCH_SIZE' + '\n'
    'or DataLoader workers will NOT help.' + '\n'
    '  1. Confirm the accelerator is "GPU T4 x2" and no other process holds GPU memory.' + '\n'
    '  2. Restart the session so both GPUs start empty, then rerun from the top.' + '\n'
    '  3. Inspect planned_device_map.json / model_load_failure_report.json.' + '\n'
    'CPU offload, disk offload, single-GPU placement and smaller backbones are all forbidden.'
)

DTYPE_ELEMENT_BYTES = torch.empty(0, dtype=BACKBONE_DTYPE).element_size()


def normalize_map_device(value):
    if isinstance(value, int):
        return 'cuda:{}'.format(value)
    text = str(value).lower().replace('cuda ', 'cuda:')
    return 'cuda:' + text if text in {'0', '1'} else text


def release_gpu_memory_before_load():
    gc.collect()
    for device_id in range(torch.cuda.device_count()):
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device_id)


def resolve_checkpoint_files(snapshot_dir):
    snapshot = Path(snapshot_dir)
    index_path = snapshot / 'model.safetensors.index.json'
    if index_path.is_file():
        names = sorted(set(read_json(index_path).get('weight_map', {}).values()))
        if not names:
            raise RuntimeError('{} contains no weight_map entries.'.format(index_path))
    else:
        available = sorted(path.name for path in snapshot.glob('*.safetensors'))
        if not available:
            raise RuntimeError('No *.safetensors weight file was found in {}.'.format(snapshot))
        names = ['model.safetensors'] if 'model.safetensors' in available else available
        if len(names) != 1:
            raise RuntimeError('Ambiguous weight files in {}: {}.'.format(snapshot, available))
    files = []
    for name in names:
        candidate = snapshot / name
        if candidate.parent != snapshot or '/' in name or name.startswith('..'):
            raise RuntimeError('Checkpoint entry {!r} escapes the snapshot directory.'.format(name))
        if not candidate.is_file():
            raise RuntimeError('Checkpoint file is missing: {}'.format(candidate))
        files.append(candidate)
    total_bytes = sum(path.stat().st_size for path in files)
    if not CHECKPOINT_MIN_TOTAL_GIB <= total_bytes / 2 ** 30 <= CHECKPOINT_MAX_TOTAL_GIB:
        raise RuntimeError('Implausible checkpoint size {:.2f} GiB.'.format(total_bytes / 2 ** 30))
    return files, total_bytes


def tensor_static_bytes(tensor):
    element_bytes = DTYPE_ELEMENT_BYTES if tensor.is_floating_point() else tensor.element_size()
    return tensor.numel() * element_bytes


def module_static_bytes(module):
    total = sum(tensor_static_bytes(p) for _, p in module.named_parameters(recurse=True))
    for _, submodule in module.named_modules():
        for buffer_name, buffer in submodule._buffers.items():
            if buffer is None or buffer_name in submodule._non_persistent_buffers_set:
                continue
            total += tensor_static_bytes(buffer)
    return total


def root_tensor_names_and_bytes(model):
    names, total = [], 0
    for name, parameter in model._parameters.items():
        if parameter is not None:
            names.append(name)
            total += tensor_static_bytes(parameter)
    for name, buffer in model._buffers.items():
        if buffer is not None and name not in model._non_persistent_buffers_set:
            names.append(name)
            total += tensor_static_bytes(buffer)
    return names, total


def plan_two_gpu_placement(model, block_attribute='blocks'):
    """Most balanced CONTIGUOUS block split, emitted as an explicit device-map dict."""
    blocks = getattr(model, block_attribute, None)
    if blocks is None or len(blocks) < 2:
        raise RuntimeError('The timm model does not expose an ordered "{}" list.'.format(block_attribute))
    num_blocks = len(blocks)
    root_names, root_bytes = root_tensor_names_and_bytes(model)
    block_bytes = [module_static_bytes(block) for block in blocks]
    input_side, output_side = [], []
    for child_name, child in model.named_children():
        if child_name == block_attribute:
            continue
        (output_side if child_name in OUTPUT_SIDE_MODULE_NAMES else input_side).append(
            (child_name, module_static_bytes(child))
        )
    input_bytes = root_bytes + sum(size for _, size in input_side)
    output_bytes = sum(size for _, size in output_side)

    candidates = []
    for split in range(1, num_blocks):
        gpu0 = input_bytes + sum(block_bytes[:split])
        gpu1 = output_bytes + sum(block_bytes[split:])
        candidates.append((max(gpu0, gpu1), gpu0, split, gpu1))
    candidates.sort()
    _, gpu0_bytes, split_index, gpu1_bytes = candidates[0]

    device_map = {name: 'cuda:0' for name in root_names}
    for child_name, _ in input_side:
        device_map[child_name] = 'cuda:0'
    for index in range(num_blocks):
        device_map['{}.{}'.format(block_attribute, index)] = 'cuda:0' if index < split_index else 'cuda:1'
    for child_name, _ in output_side:
        device_map[child_name] = 'cuda:1'
    if '' in device_map:
        raise RuntimeError('The device map must not contain a root "" key.')
    if set(device_map.values()) != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('The planned device map does not span both GPUs.')

    return {
        'strategy': SHARDING_STRATEGY,
        'transformer_block_count': num_blocks,
        'contiguous_split_index': split_index,
        'cuda_0_block_range': [0, split_index - 1],
        'cuda_1_block_range': [split_index, num_blocks - 1],
        'estimated_static_bytes': {'cuda:0': int(gpu0_bytes), 'cuda:1': int(gpu1_bytes)},
        'estimated_static_gib': {'cuda:0': gpu0_bytes / 2 ** 30, 'cuda:1': gpu1_bytes / 2 ** 30},
        'device_map': device_map,
    }


def enforce_placement_budget(plan):
    limit_bytes = GPU_STATIC_MEMORY_LIMIT_GIB * 2 ** 30
    budget_rows = []
    for device_id in (0, 1):
        device_key = 'cuda:{}'.format(device_id)
        planned_bytes = plan['estimated_static_bytes'][device_key]
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        headroom_gib = (total_bytes - planned_bytes) / 2 ** 30
        budget_rows.append({
            'device': device_key,
            'planned_static_gib': planned_bytes / 2 ** 30,
            'device_free_gib': free_bytes / 2 ** 30,
            'projected_headroom_gib': headroom_gib,
        })
        if planned_bytes > limit_bytes:
            raise RuntimeError('Planned static memory on {} exceeds the ceiling.'.format(device_key)
                               + '\n' + DISPATCH_FAILURE_HINT)
        if headroom_gib < GPU_MIN_RUNTIME_HEADROOM_GIB:
            raise RuntimeError('Runtime headroom on {} too small.'.format(device_key)
                               + '\n' + DISPATCH_FAILURE_HINT)
        if free_bytes < planned_bytes:
            raise RuntimeError('{} has only {:.2f} GiB free; restart the session.'.format(
                device_key, free_bytes / 2 ** 30) + '\n' + DISPATCH_FAILURE_HINT)
    plan['budget_rows'] = budget_rows
    return plan


def device_for_tensor_name(tensor_name, device_map):
    """Longest-prefix module match, mirroring Accelerate's own resolution order."""
    best_name, best_device = None, None
    for module_name, device in device_map.items():
        if tensor_name == module_name or tensor_name.startswith(module_name + '.'):
            if best_name is None or len(module_name) > len(best_name):
                best_name, best_device = module_name, device
    if best_device is None:
        raise RuntimeError('No device assignment covers checkpoint tensor {!r}.'.format(tensor_name))
    return best_device


def stream_checkpoint_into_model(model, checkpoint_files, device_map, dtype):
    """One tensor at a time: read on host, cast to FP16 on host, send only to its target GPU."""
    model_tensor_names = set(model.state_dict().keys())
    checkpoint_names, loaded = set(), 0
    for checkpoint_file in checkpoint_files:
        with safe_open(str(checkpoint_file), framework='pt', device='cpu') as handle:
            keys = sorted(handle.keys())
            checkpoint_names.update(keys)
            for key in tqdm(keys, desc='streaming {}'.format(checkpoint_file.name)):
                if key not in model_tensor_names:
                    continue
                tensor = handle.get_tensor(key)
                if tensor.is_floating_point():
                    tensor = tensor.to(dtype)
                set_module_tensor_to_device(model, key, device_for_tensor_name(key, device_map),
                                            value=tensor, dtype=dtype)
                del tensor
                loaded += 1
    return {
        'tensors_loaded': loaded,
        'missing_keys': sorted(model_tensor_names - checkpoint_names),
        'unexpected_keys': sorted(checkpoint_names - model_tensor_names),
    }


def build_timm_backbone(snapshot_dir):
    with init_empty_weights(include_buffers=False):
        model = timm.create_model(TIMM_MODEL_NAME, pretrained=False, num_classes=0, img_size=IMAGE_SIZE)
    blocks = getattr(model, 'blocks', None)
    if blocks is None or len(blocks) < 2:
        raise RuntimeError('Could not resolve an ordered transformer block list.')
    block_class_name = type(blocks[0]).__name__
    checkpoint_files, checkpoint_bytes = resolve_checkpoint_files(snapshot_dir)
    plan = plan_two_gpu_placement(model)
    plan['checkpoint_total_gib'] = checkpoint_bytes / 2 ** 30
    release_gpu_memory_before_load()
    plan['memory_before_load'] = cuda_memory_snapshot('before_checkpoint_load')
    write_json_atomic(plan, ARTIFACT_DIR / 'planned_device_map.json')
    plan = enforce_placement_budget(plan)
    write_json_atomic(plan, ARTIFACT_DIR / 'planned_device_map.json')

    print('Planned contiguous split: blocks {}-{} -> cuda:0, blocks {}-{} -> cuda:1'.format(
        plan['cuda_0_block_range'][0], plan['cuda_0_block_range'][1],
        plan['cuda_1_block_range'][0], plan['cuda_1_block_range'][1],
    ))
    try:
        load_report = stream_checkpoint_into_model(model, checkpoint_files, plan['device_map'], BACKBONE_DTYPE)
        if load_report['missing_keys']:
            raise RuntimeError('Checkpoint missing {} model tensors, e.g. {}.'.format(
                len(load_report['missing_keys']), load_report['missing_keys'][:10]))
        remaining_meta = [
            name for name, tensor in list(model.named_parameters()) + list(model.named_buffers())
            if tensor.device.type == 'meta'
        ]
        if remaining_meta:
            raise RuntimeError('These tensors are still on meta: {}.'.format(remaining_meta[:20]))
        model = dispatch_model(model, device_map=plan['device_map'], force_hooks=True)
    except Exception as exc:
        write_json_atomic(
            {'stage': 'timm_checkpoint_streaming_and_dispatch', 'error_type': type(exc).__name__,
             'error': str(exc), 'planned_device_map': plan['device_map'],
             'memory_after_failure': cuda_memory_snapshot('after_failure'), 'action': DISPATCH_FAILURE_HINT},
            ARTIFACT_DIR / 'model_load_failure_report.json',
        )
        raise RuntimeError(DISPATCH_FAILURE_HINT) from exc

    model.eval().requires_grad_(False)
    plan['load_report'] = load_report
    write_json_atomic(plan, ARTIFACT_DIR / 'planned_device_map.json')
    return model, list(model.blocks), block_class_name, plan


def audit_dispatched_backbone(model, blocks):
    raw_device_map = getattr(model, 'hf_device_map', None)
    if not isinstance(raw_device_map, dict) or not raw_device_map:
        raise RuntimeError('Accelerate did not attach an hf_device_map.' + '\n' + DISPATCH_FAILURE_HINT)
    device_map = {name: normalize_map_device(value) for name, value in raw_device_map.items()}
    mapped_devices = set(device_map.values())
    forbidden = sorted(v for v in mapped_devices if 'cpu' in v or 'disk' in v or 'meta' in v)
    if forbidden:
        raise RuntimeError('CPU/disk/meta placement detected: {}.'.format(forbidden))
    if mapped_devices != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('Device map must span exactly cuda:0 and cuda:1; got {}.'.format(sorted(mapped_devices)))
    meta_tensors = [n for n, t in list(model.named_parameters()) + list(model.named_buffers())
                    if t.device.type == 'meta']
    if meta_tensors:
        raise RuntimeError('Tensors still on meta: {}.'.format(meta_tensors[:20]))
    if {str(p.device) for p in model.parameters()} != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('Parameters are not split across exactly both GPUs.')
    if {p.dtype for p in model.parameters() if p.is_floating_point()} != {BACKBONE_DTYPE}:
        raise RuntimeError('Floating parameters are not uniformly FP16.')

    block_devices = []
    for block_index, block in enumerate(blocks):
        devices = {str(p.device) for p in block.parameters()}
        if len(devices) != 1:
            raise RuntimeError('Block {} is split across devices {}.'.format(block_index, sorted(devices)))
        block_devices.append(devices.pop())
    transitions = [i for i in range(1, len(block_devices)) if block_devices[i] != block_devices[i - 1]]
    if len(transitions) != 1:
        raise RuntimeError('Blocks must form exactly two contiguous device ranges; found {}.'.format(
            len(transitions)))
    if block_devices[0] != 'cuda:0' or block_devices[-1] != 'cuda:1':
        raise RuntimeError('Contiguous block ranges must run cuda:0 then cuda:1.')

    hooked = [n for n, m in model.named_modules() if hasattr(m, '_hf_hook')]
    if not hooked:
        raise RuntimeError('No Accelerate hooks are attached.' + '\n' + DISPATCH_FAILURE_HINT)
    for device_id in (0, 1):
        free_bytes, _ = torch.cuda.mem_get_info(device_id)
        if free_bytes / 2 ** 30 < GPU_MIN_RUNTIME_HEADROOM_GIB:
            raise RuntimeError('cuda:{} has only {:.2f} GiB free after loading.'.format(
                device_id, free_bytes / 2 ** 30))
    forbidden_wrappers = {''.join(('Data', 'Parallel')), ''.join(('Distributed', 'Data', 'Parallel')),
                          ''.join(('Fully', 'Sharded', 'Data', 'Parallel'))}
    if type(model).__name__ in forbidden_wrappers:
        raise RuntimeError('Replication wrapper detected; only model sharding is allowed.')
    return device_map, block_devices, len(hooked)


MODEL_LOAD_STARTED_AT = time.perf_counter()
BACKBONE, BACKBONE_BLOCKS, BACKBONE_BLOCK_CLASS, PLACEMENT_PLAN = build_timm_backbone(MODEL_SNAPSHOT_DIR)
MODEL_LOAD_SECONDS = time.perf_counter() - MODEL_LOAD_STARTED_AT
del HF_TOKEN_VALUE

DEVICE_MAP, BLOCK_DEVICES, HOOKED_MODULE_COUNT = audit_dispatched_backbone(BACKBONE, BACKBONE_BLOCKS)
BACKBONE_PARAMETER_COUNT = sum(p.numel() for p in BACKBONE.parameters())
if sum(p.numel() for p in BACKBONE.parameters() if p.requires_grad) != 0:
    raise RuntimeError('The pretrained backbone is not completely frozen.')
if BACKBONE.training:
    raise RuntimeError('The backbone must stay in evaluation mode.')

NUM_PREFIX_TOKENS = getattr(BACKBONE, 'num_prefix_tokens', None)
if NUM_PREFIX_TOKENS is None:
    raise RuntimeError('The timm model does not expose num_prefix_tokens.')
NUM_PREFIX_TOKENS = int(NUM_PREFIX_TOKENS)
register_parameter = getattr(BACKBONE, 'reg_token', None)
NUM_REGISTER_TOKENS = int(register_parameter.shape[1]) if register_parameter is not None else NUM_PREFIX_TOKENS - 1
HIDDEN_SIZE = int(BACKBONE.num_features)
FINAL_NORM_MODULE = BACKBONE.norm
MODEL_INPUT_DEVICE = next(BACKBONE.patch_embed.proj.parameters()).device
FINAL_NORM_DEVICE = next(FINAL_NORM_MODULE.parameters()).device
SEQUENCE_TOKEN_COUNT = NUM_PREFIX_TOKENS + EXPECTED_PATCH_TOKENS

if HIDDEN_SIZE != EXPECTED_HIDDEN_SIZE:
    raise RuntimeError('Runtime hidden size {} != expected {}.'.format(HIDDEN_SIZE, EXPECTED_HIDDEN_SIZE))
if NUM_PREFIX_TOKENS != EXPECTED_PREFIX_TOKENS or NUM_REGISTER_TOKENS != EXPECTED_REGISTER_TOKENS:
    raise RuntimeError('Runtime prefix/register token counts ({}, {}) != expected.'.format(
        NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS))
if SEQUENCE_TOKEN_COUNT != EXPECTED_SEQUENCE_TOKENS:
    raise RuntimeError('Runtime sequence token count {} != expected.'.format(SEQUENCE_TOKEN_COUNT))
if str(FINAL_NORM_DEVICE) != BLOCK_DEVICES[-1]:
    raise RuntimeError('The final norm is not on the same GPU as the last transformer block.')

TIMM_DATA_CONFIG = timm.data.resolve_model_data_config(BACKBONE)
if not np.allclose(tuple(float(v) for v in TIMM_DATA_CONFIG['mean']), IMAGENET_MEAN, atol=1e-6):
    raise RuntimeError('timm normalization does not match the locked ImageNet constants.')
if tuple(int(v) for v in TIMM_DATA_CONFIG['input_size']) != (3, IMAGE_SIZE, IMAGE_SIZE):
    raise RuntimeError('timm input size is not the native {}px contract.'.format(IMAGE_SIZE))

MODEL_MANIFEST = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'backend': 'timm',
    'repository': TIMM_HF_REPO_ID,
    'immutable_revision': RESOLVED_REVISION,
    'revision_matches_v10': MODEL_REVISION_MATCHES_V10,
    'timm_model_name': TIMM_MODEL_NAME,
    'model_class': type(BACKBONE).__name__,
    'block_class': BACKBONE_BLOCK_CLASS,
    'parameter_count': BACKBONE_PARAMETER_COUNT,
    'trainable_parameter_count': 0,
    'transformer_block_count': len(BACKBONE_BLOCKS),
    'hidden_size': HIDDEN_SIZE,
    'num_prefix_tokens': NUM_PREFIX_TOKENS,
    'num_register_tokens': NUM_REGISTER_TOKENS,
    'sequence_token_count': SEQUENCE_TOKEN_COUNT,
    'token_layout': 'CLS, register tokens, true patch tokens',
    'backbone_dtype': BACKBONE_DTYPE_NAME,
    'placement_strategy': SHARDING_STRATEGY,
    'block_devices': BLOCK_DEVICES,
    'accelerate_hooked_module_count': HOOKED_MODULE_COUNT,
    'automatic_device_map_string_used': False,
    'cpu_or_disk_offload': False,
    'data_parallel_used': False,
    'model_download_seconds': MODEL_DOWNLOAD_SECONDS,
    'model_load_seconds': MODEL_LOAD_SECONDS,
    'timm_resolved_data_config': {
        'mean': [float(v) for v in TIMM_DATA_CONFIG['mean']],
        'std': [float(v) for v in TIMM_DATA_CONFIG['std']],
        'input_size': [int(v) for v in TIMM_DATA_CONFIG['input_size']],
        'interpolation': TIMM_DATA_CONFIG.get('interpolation'),
        'crop_pct': TIMM_DATA_CONFIG.get('crop_pct'),
    },
    'applied_interpolation': INTERPOLATION_NAME,
    'interpolation_note': V13_NOTE,
    'backbone_documented_for_final_report': (
        'DINOv3 ViT-7B/16, timm repo timm/vit_7b_patch16_dinov3.lvd1689m, LVD-1689M pretraining checkpoint, '
        'revision {}, 100 percent frozen in FP16, sharded across two Tesla T4 GPUs. Only a small classifier '
        'head was trained, on the official BDC Satria Data 2026 training images. No external labeled images '
        'and no test labels were used.'.format(RESOLVED_REVISION)
    ),
    'environment': ENVIRONMENT_INFO,
    'gpus': GPU_RUNTIME_INFO,
}
write_json_atomic(MODEL_MANIFEST, ARTIFACT_DIR / 'model_manifest.json')
write_json_atomic({'device_map': DEVICE_MAP, 'block_devices': BLOCK_DEVICES,
                   'actual_cuda_memory': cuda_memory_snapshot('after_audit')},
                  ARTIFACT_DIR / 'device_map.json')

print()
print('Backbone        :', type(BACKBONE).__name__, '| block class', BACKBONE_BLOCK_CLASS)
print('Parameters      : {:,} (trainable 0)'.format(BACKBONE_PARAMETER_COUNT))
print('Block placement : cuda:0 x{} then cuda:1 x{}'.format(
    BLOCK_DEVICES.count('cuda:0'), BLOCK_DEVICES.count('cuda:1')))
print('Token contract  : {} = {} prefix ({} register) + {} patch, hidden {}'.format(
    SEQUENCE_TOKEN_COUNT, NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS, EXPECTED_PATCH_TOKENS, HIDDEN_SIZE))
for row in cuda_memory_snapshot('after_audit'):
    print('  cuda:{device_id} allocated {allocated_gib:.2f} GiB, free {free_gib:.2f} GiB'.format(**row))

In [ ]:
# ---------------------------------------------------------------------------
# THE FIXED RANDOM PROJECTION
# Generated once from a fixed seed on CPU and hashed, so train and test share exactly the
# same P. It is materialised per device on demand rather than pinned to one GPU: timm's
# forward_features is a different method from forward, so Accelerate's root hook does not
# post-process its output and the token device is an implementation detail we must not assume.
# ---------------------------------------------------------------------------
_gram_generator = torch.Generator().manual_seed(GRAM_PROJECTION_SEED)
GRAM_PROJECTION = torch.randn(
    HIDDEN_SIZE, GRAM_PROJECTION_DIM, generator=_gram_generator, dtype=torch.float32
) / math.sqrt(GRAM_PROJECTION_DIM)
GRAM_PROJECTION_SHA256 = hashlib.sha256(GRAM_PROJECTION.numpy().tobytes()).hexdigest()
GRAM_PROJECTION_BY_DEVICE = {}
del _gram_generator


def gram_projection_for(device):
    key = str(device)
    if key not in GRAM_PROJECTION_BY_DEVICE:
        GRAM_PROJECTION_BY_DEVICE[key] = GRAM_PROJECTION.to(device=device, dtype=POOLING_COMPUTE_DTYPE)
    return GRAM_PROJECTION_BY_DEVICE[key]


def normalized_tokens(model_input):
    """timm forward_features ends with self.norm, so the official final norm is applied exactly once.

    V13 proved this path is numerically identical to V10's forward_intermediates(norm=True) route:
    its in-session control reproduced the champion with macro_f1_delta = 0.0 and identical errors.
    """
    tokens = BACKBONE.forward_features(model_input)
    if not torch.is_tensor(tokens):
        raise RuntimeError('forward_features did not return a token tensor.')
    if tokens.ndim != 3 or tokens.shape[1] != SEQUENCE_TOKEN_COUNT or tokens.shape[2] != HIDDEN_SIZE:
        raise RuntimeError('forward_features returned shape {}, expected [B, {}, {}].'.format(
            tuple(tokens.shape), SEQUENCE_TOKEN_COUNT, HIDDEN_SIZE))
    if tokens.dtype != BACKBONE_DTYPE:
        raise RuntimeError('Normalized tokens are not FP16 ({}).'.format(tokens.dtype))
    return tokens


def pool_first_and_second_order(tokens):
    """[B, 261, 4096] normalised tokens -> ([B, 2, 4096] first order, [B, D, D] Gram), both FP32.

    First order  : prefix token 0 (CLS) and the mean of the 256 true patch tokens.
                   Register tokens sit at prefix positions 1..4 and are sliced away.
    Second order : patch tokens L2-normalised per token (the DINOv3 Gram-anchoring formulation),
                   projected by the fixed random P, then C = Y^T Y / num_patches.
    """
    cls_fp32 = tokens[:, 0, :].to(POOLING_COMPUTE_DTYPE)
    patches_fp32 = tokens[:, NUM_PREFIX_TOKENS:, :].to(POOLING_COMPUTE_DTYPE)
    num_patches = patches_fp32.shape[1]
    if num_patches != EXPECTED_PATCH_TOKENS:
        raise RuntimeError('True-patch token count is {}, expected {}.'.format(
            num_patches, EXPECTED_PATCH_TOKENS))
    first_order = torch.stack((cls_fp32, patches_fp32.mean(dim=1)), dim=1)

    unit_patches = F.normalize(patches_fp32, dim=-1, eps=1e-6)
    projected = torch.matmul(unit_patches, gram_projection_for(unit_patches.device))
    gram = torch.matmul(projected.transpose(1, 2), projected) / float(num_patches)
    gram = 0.5 * (gram + gram.transpose(1, 2))   # kill FP asymmetry; C is symmetric by construction
    return first_order, gram


def _finalize(tensor, expected_tail, expected_rows):
    if tensor.dtype != POOLING_COMPUTE_DTYPE:
        raise RuntimeError('Feature storage expects FP32, got {}.'.format(tensor.dtype))
    if tuple(tensor.shape) != (expected_rows,) + tuple(expected_tail):
        raise RuntimeError('Feature shape {} is invalid, expected {}.'.format(
            tuple(tensor.shape), (expected_rows,) + tuple(expected_tail)))
    if not torch.isfinite(tensor).all().item():
        raise RuntimeError('Features contain NaN or Inf; extraction is aborted.')
    max_abs = float(tensor.abs().max().item())
    if not math.isfinite(max_abs) or max_abs > FEATURE_MAX_ABS_LIMIT:
        raise RuntimeError('Feature range is implausible: {}.'.format(max_abs))
    return tensor.detach().to(device='cpu', dtype=FEATURE_STORAGE_DTYPE)


def extract_features(pixel_values):
    """[B, 3, 256, 256] FP32 on CPU -> ([B, 2, 4096], [B, D, D]) FP16 on CPU."""
    if pixel_values.ndim != 4 or tuple(pixel_values.shape[1:]) != (3, IMAGE_SIZE, IMAGE_SIZE):
        raise RuntimeError('Feature extraction expects [B, 3, {}, {}], got {}.'.format(
            IMAGE_SIZE, IMAGE_SIZE, tuple(pixel_values.shape)))
    if pixel_values.dtype != torch.float32:
        raise RuntimeError('Feature extraction expects FP32 pixel values, got {}.'.format(pixel_values.dtype))
    rows = pixel_values.shape[0]
    with torch.inference_mode():
        model_input = pixel_values.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE, non_blocking=True)
        first_order, gram = pool_first_and_second_order(normalized_tokens(model_input))
        return (
            _finalize(first_order, (len(FIRST_ORDER_BRANCHES), HIDDEN_SIZE), rows),
            _finalize(gram, (GRAM_PROJECTION_DIM, GRAM_PROJECTION_DIM), rows),
        )


def deterministic_pattern(width, height, phase):
    x = np.arange(width, dtype=np.uint32)[None, :]
    y = np.arange(height, dtype=np.uint32)[:, None]
    red = np.broadcast_to((x * 7 + phase * 31) % 256, (height, width))
    green = np.broadcast_to((y * 11 + phase * 17) % 256, (height, width))
    checker = (((x // 9 + y // 13 + phase) % 2) * 127).astype(np.uint32)
    blue = (x * 3 + y * 5 + checker + phase * 23) % 256
    return Image.fromarray(np.stack((red, green, blue), axis=2).astype(np.uint8))


def deterministic_sample_paths(frame, count):
    ranked = sorted(
        (hashlib.sha256(image_id.encode('utf-8')).hexdigest(), path)
        for image_id, path in zip(frame['image_id'], frame['path'])
    )
    if count > len(ranked):
        raise RuntimeError('Sample is larger than the indexed train set.')
    return [path for _, path in ranked[:count]]


def run_preflight():
    stage = 'A_token_contract'
    report = {
        'schema_version': 1, 'revision_marker': REVISION_MARKER, 'pass': False, 'failed_stage': None,
        'model_revision': RESOLVED_REVISION, 'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
        'gram_projection_sha256': GRAM_PROJECTION_SHA256, 'stages': {},
    }
    try:
        synthetic = [deterministic_pattern(317, 193, 1), deterministic_pattern(211, 359, 2)]
        synthetic_batch = torch.stack([IMAGE_TRANSFORM(image) for image in synthetic])
        with torch.inference_mode():
            model_input = synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE)
            tokens = normalized_tokens(model_input)
            token_device = str(tokens.device)
            tokens_finite = bool(torch.isfinite(tokens.float()).all().item())
            double_normed = FINAL_NORM_MODULE(tokens.to(FINAL_NORM_DEVICE))
            double_norm_delta = float(
                (double_normed.float() - tokens.to(FINAL_NORM_DEVICE).float()).abs().max().item()
            )
        report['stages'][stage] = {
            'pass': bool(tokens_finite and double_norm_delta > 0.0),
            'sequence_token_count': int(tokens.shape[1]), 'prefix_token_count': NUM_PREFIX_TOKENS,
            'register_token_count': NUM_REGISTER_TOKENS, 'hidden_size': int(tokens.shape[2]),
            'token_device': token_device, 'final_norm_device': str(FINAL_NORM_DEVICE),
            'second_norm_application_max_abs_delta': double_norm_delta, 'final_norm_applied_times': 1,
        }
        if not report['stages'][stage]['pass']:
            raise RuntimeError('Token contract or final-norm-once check failed.')

        stage = 'B_first_order_matches_champion'
        with torch.inference_mode():
            tokens = normalized_tokens(synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE))
            first_order, gram = pool_first_and_second_order(tokens)
            direct_cls = tokens[:, 0, :].to(POOLING_COMPUTE_DTYPE)
            true_patch_mean = tokens[:, NUM_PREFIX_TOKENS:, :].to(POOLING_COMPUTE_DTYPE).mean(dim=1)
            all_non_cls_mean = tokens[:, 1:, :].to(POOLING_COMPUTE_DTYPE).mean(dim=1)
            cls_delta = float((first_order[:, 0] - direct_cls).abs().max().item())
            mean_delta = float((first_order[:, 1] - true_patch_mean).abs().max().item())
            register_delta = float((true_patch_mean - all_non_cls_mean).abs().max().item())
        first_order_ok = cls_delta == 0.0 and mean_delta == 0.0 and register_delta > 0.0
        report['stages'][stage] = {
            'pass': bool(first_order_ok),
            'branch0_vs_direct_cls_max_abs_delta': cls_delta,
            'branch1_vs_direct_true_patch_mean_max_abs_delta': mean_delta,
            'register_exclusion_max_abs_delta': register_delta,
            'note': 'the first-order branch must be byte-for-byte the V10 champion readout',
        }
        if not first_order_ok:
            raise RuntimeError('The first-order branch does not match the champion readout exactly.')

        stage = 'C_gram_sufficiency_identity'
        # The entire cache design rests on W^T (Y^T Y) W == (Y W)^T (Y W). Verify it numerically
        # rather than trusting the algebra, because a violation would silently invalidate the head.
        with torch.inference_mode():
            tokens = normalized_tokens(synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE))
            patches_fp32 = tokens[:, NUM_PREFIX_TOKENS:, :].to(POOLING_COMPUTE_DTYPE)
            unit_patches = F.normalize(patches_fp32, dim=-1, eps=1e-6)
            projected = torch.matmul(unit_patches, gram_projection_for(unit_patches.device))
            cached_gram = torch.matmul(projected.transpose(1, 2), projected) / float(patches_fp32.shape[1])
            probe_generator = torch.Generator().manual_seed(7)
            probe_w = torch.randn(
                GRAM_PROJECTION_DIM, GRAM_RANK, generator=probe_generator, dtype=torch.float32
            ).to(cached_gram.device)
            via_cache = torch.matmul(probe_w.t(), torch.matmul(cached_gram, probe_w))
            direct_z = torch.matmul(projected, probe_w)
            via_tokens = torch.matmul(direct_z.transpose(1, 2), direct_z) / float(patches_fp32.shape[1])
            identity_delta = float((via_cache - via_tokens).abs().max().item())
            reference_scale = float(via_tokens.abs().max().item())
        identity_ok = identity_delta <= max(1e-4, 1e-3 * reference_scale)
        report['stages'][stage] = {
            'pass': bool(identity_ok),
            'max_abs_delta': identity_delta,
            'reference_scale': reference_scale,
            'note': 'W^T C W from the cached Gram must equal (YW)^T(YW) computed from the tokens',
        }
        if not identity_ok:
            raise RuntimeError(
                'Gram sufficiency identity failed (delta {:.3e}). Caching C would not be lossless for '
                'the head family used here.'.format(identity_delta))

        stage = 'D_gram_carries_signal'
        # V13's lesson: measure whether the new quantity actually varies across images BEFORE
        # spending the extraction budget. There, feature cosine was 0.99941 and the experiment was
        # dead on arrival. Same early-warning check, applied to the Gram descriptor.
        probe_paths = deterministic_sample_paths(train_df, 64)
        probe_first, probe_gram = [], []
        for start in range(0, len(probe_paths), FEATURE_BATCH_SIZE):
            batch = torch.stack([
                IMAGE_TRANSFORM(load_rgb_image(path))
                for path in probe_paths[start:start + FEATURE_BATCH_SIZE]
            ])
            first_chunk, gram_chunk = extract_features(batch)
            probe_first.append(first_chunk.to(POOLING_COMPUTE_DTYPE))
            probe_gram.append(gram_chunk.to(POOLING_COMPUTE_DTYPE))
        probe_first = torch.cat(probe_first)
        probe_gram = torch.cat(probe_gram)

        triu_rows, triu_cols = torch.triu_indices(GRAM_PROJECTION_DIM, GRAM_PROJECTION_DIM)
        gram_unit = F.normalize(probe_gram[:, triu_rows, triu_cols], dim=1)
        gram_pairwise = torch.matmul(gram_unit, gram_unit.t())
        off_diag = ~torch.eye(len(probe_paths), dtype=torch.bool)
        gram_mean_pairwise_cosine = float(gram_pairwise[off_diag].mean().item())

        first_unit = F.normalize(probe_first.flatten(1), dim=1)
        first_pairwise = torch.matmul(first_unit, first_unit.t())
        first_mean_pairwise_cosine = float(first_pairwise[off_diag].mean().item())

        diagonal_energy = float(torch.diagonal(probe_gram, dim1=1, dim2=2).pow(2).sum().item())
        total_energy = float(probe_gram.pow(2).sum().item())
        off_diagonal_share = 1.0 - diagonal_energy / max(total_energy, 1e-12)

        gram_informative = gram_mean_pairwise_cosine < 0.999 and off_diagonal_share > 0.01
        report['stages'][stage] = {
            'pass': bool(gram_informative),
            'probe_images': len(probe_paths),
            'gram_mean_pairwise_cosine': gram_mean_pairwise_cosine,
            'first_order_mean_pairwise_cosine': first_mean_pairwise_cosine,
            'gram_off_diagonal_energy_share': off_diagonal_share,
            'v13_dead_on_arrival_reference': 0.99941,
            'note': (
                'lower pairwise cosine means the descriptor separates images more. A Gram cosine at or '
                'above 0.999 would mean the second-order descriptor is near-constant and the experiment '
                'is dead on arrival, exactly the failure mode V13 hit at 0.99941.'
            ),
        }
        if not gram_informative:
            raise RuntimeError(
                'The Gram descriptor is near-constant across images (mean pairwise cosine {:.6f}, '
                'off-diagonal energy share {:.4f}). Second-order pooling cannot carry signal here; '
                'stop before spending the extraction budget.'.format(
                    gram_mean_pairwise_cosine, off_diagonal_share))

        stage = 'E_throughput'
        for device_id in (0, 1):
            torch.cuda.reset_peak_memory_stats(device_id)
        sample_paths = deterministic_sample_paths(train_df, BENCHMARK_WARMUP_IMAGES + BENCHMARK_TIMED_IMAGES)
        warmup_loader = DataLoader(ImageRows(sample_paths[:BENCHMARK_WARMUP_IMAGES]),
                                   batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0)
        for batch in warmup_loader:
            extract_features(batch)
        for device_id in (0, 1):
            torch.cuda.synchronize(device_id)
        timed_loader = DataLoader(ImageRows(sample_paths[BENCHMARK_WARMUP_IMAGES:]),
                                  batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0)
        timed_started = time.perf_counter()
        for batch in timed_loader:
            extract_features(batch)
        for device_id in (0, 1):
            torch.cuda.synchronize(device_id)
        timed_seconds = time.perf_counter() - timed_started
        images_per_second = BENCHMARK_TIMED_IMAGES / timed_seconds
        report['stages'][stage] = {
            'pass': bool(images_per_second >= MIN_EXTRACTION_IMAGES_PER_SECOND),
            'timed_seconds': timed_seconds, 'images_per_second': images_per_second,
            'estimated_train_hours': EXPECTED_TRAIN_ROWS / images_per_second / 3600.0,
            'estimated_test_minutes': EXPECTED_TEST_ROWS / images_per_second / 60.0,
            'v13_measured_images_per_second': EXPECTED_TRAIN_ROWS / 1.3835663795093216 / 3600.0,
            'gpu_memory': gpu_memory_rows(), 'host_rss_gib': host_rss_gib(),
        }
        if not report['stages'][stage]['pass']:
            raise RuntimeError('Measured {:.3f} images/s is below the locked gate.'.format(images_per_second))

        report['pass'] = True
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        return report
    except torch.cuda.OutOfMemoryError as exc:
        report.update({'failed_stage': stage, 'error_type': type(exc).__name__, 'error': str(exc),
                       'action': 'Lower FEATURE_BATCH_SIZE (try 2, then 1) and rerun from the top.'})
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        raise RuntimeError('GPU OOM during preflight stage {} at FEATURE_BATCH_SIZE={}.'.format(
            stage, FEATURE_BATCH_SIZE)) from exc
    except Exception as exc:
        report.update({'failed_stage': stage, 'error_type': type(exc).__name__, 'error': str(exc),
                       'action': 'Resolve the reported stage and rerun. Full extraction has not started.'})
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        raise RuntimeError('Preflight failed at stage {}; inspect {}.'.format(stage, PREFLIGHT_REPORT_PATH)) from exc


PREFLIGHT_REPORT = run_preflight()
PREFLIGHT_SIGNATURE = sha256_json({
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'model_revision': RESOLVED_REVISION,
    'gram_projection_sha256': GRAM_PROJECTION_SHA256,
    'token_contract': [SEQUENCE_TOKEN_COUNT, NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS,
                       EXPECTED_PATCH_TOKENS, HIDDEN_SIZE],
})

signal_stage = PREFLIGHT_REPORT['stages']['D_gram_carries_signal']
throughput_stage = PREFLIGHT_REPORT['stages']['E_throughput']
print()
print('Preflight PASSED. Signature:', PREFLIGHT_SIGNATURE)
print('Projection sha256        :', GRAM_PROJECTION_SHA256)
print('Token device             :', PREFLIGHT_REPORT['stages']['A_token_contract']['token_device'],
      '(projection follows it automatically)')
print('First-order branch       : byte-identical to the V10 champion readout')
print('Gram sufficiency identity: max abs delta {:.3e}  (W^T C W == (YW)^T(YW))'.format(
    PREFLIGHT_REPORT['stages']['C_gram_sufficiency_identity']['max_abs_delta']))
print('Gram carries signal      : mean pairwise cosine {:.6f} (first-order {:.6f}); off-diag energy {:.4f}'.format(
    signal_stage['gram_mean_pairwise_cosine'], signal_stage['first_order_mean_pairwise_cosine'],
    signal_stage['gram_off_diagonal_energy_share']))
print('   V13 died at 0.99941 on this style of check; below 0.999 here means a live experiment.')
print('Throughput               : {:.3f} img/s -> train ~{:.2f} h, test ~{:.1f} min'.format(
    throughput_stage['images_per_second'], throughput_stage['estimated_train_hours'],
    throughput_stage['estimated_test_minutes']))

In [ ]:
CACHE_SCHEMA_VERSION = 1
FIRST_ORDER_TAIL = (len(FIRST_ORDER_BRANCHES), HIDDEN_SIZE)
GRAM_TAIL = (GRAM_PROJECTION_DIM, GRAM_PROJECTION_DIM)


def build_cache_identity(split, row_signature):
    identity = {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'identity_scope': 'frozen_backbone_features_only_no_head_hyperparameters',
        'backend': 'timm',
        'model_repository': TIMM_HF_REPO_ID,
        'model_revision': RESOLVED_REVISION,
        'timm_model_name': TIMM_MODEL_NAME,
        'timm_version': ENVIRONMENT_INFO['timm_version'],
        'torch_version': ENVIRONMENT_INFO['torch_version'],
        'cuda_version': ENVIRONMENT_INFO['cuda_version'],
        'gpu_names': [info['name'] for info in GPU_RUNTIME_INFO],
        'image_size': IMAGE_SIZE,
        'interpolation': INTERPOLATION_NAME,
        'preprocess_signature': PREPROCESS_SIGNATURE,
        'backbone_dtype': BACKBONE_DTYPE_NAME,
        'pooling_compute_dtype': POOLING_COMPUTE_DTYPE_NAME,
        'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
        'first_order_branches': list(FIRST_ORDER_BRANCHES),
        'gram_projection_dim': GRAM_PROJECTION_DIM,
        'gram_projection_seed': GRAM_PROJECTION_SEED,
        'gram_projection_sha256': GRAM_PROJECTION_SHA256,
        'gram_definition': 'C = Y^T Y / num_patches with Y = L2_normalize(patch tokens) @ P',
        'token_contract': {
            'sequence_token_count': SEQUENCE_TOKEN_COUNT,
            'prefix_token_count': NUM_PREFIX_TOKENS,
            'register_token_count': NUM_REGISTER_TOKENS,
            'patch_token_count': EXPECTED_PATCH_TOKENS,
            'hidden_size': HIDDEN_SIZE,
            'register_tokens_excluded': True,
        },
        'raw_patch_tokens_stored': False,
        'preflight_signature': PREFLIGHT_SIGNATURE,
        'split': split,
    }
    identity.update(row_signature)
    return identity


def expected_shard_metadata(split, cache_signature, frame, start, end):
    window = frame.iloc[start:end]
    labels = window['label'].astype(int).tolist() if 'label' in window.columns else None
    folds = window['fold'].astype(int).tolist() if 'fold' in window.columns else None
    return {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'cache_signature': cache_signature,
        'split': split,
        'start': int(start),
        'end': int(end),
        'rows': int(end - start),
        'image_ids_sha256': sha256_json(window['image_id'].tolist()),
        'labels_sha256': sha256_json(labels) if labels is not None else None,
        'folds_sha256': sha256_json(folds) if folds is not None else None,
        'first_order_shape': [int(end - start)] + list(FIRST_ORDER_TAIL),
        'gram_shape': [int(end - start)] + list(GRAM_TAIL),
        'feature_dtype': FEATURE_STORAGE_DTYPE_NAME,
    }


def validate_shard(path, expected_metadata, frame, start, end):
    """Every failure raises RuntimeError so the caller can safely recompute the shard."""
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
    except Exception as exc:
        raise RuntimeError('Feature shard cannot be loaded: {}'.format(path)) from exc
    if not isinstance(payload, dict) or payload.get('metadata') != expected_metadata:
        raise RuntimeError('Feature shard metadata is stale or mismatched: {}'.format(path))
    for key, expected_shape in (('first_order', expected_metadata['first_order_shape']),
                                ('gram', expected_metadata['gram_shape'])):
        tensor = payload.get(key)
        if not torch.is_tensor(tensor) or list(tensor.shape) != list(expected_shape):
            raise RuntimeError('Shard tensor "{}" has an invalid shape: {}'.format(key, path))
        if tensor.dtype != FEATURE_STORAGE_DTYPE:
            raise RuntimeError('Shard tensor "{}" has an invalid dtype: {}'.format(key, path))
        if not torch.isfinite(tensor.to(POOLING_COMPUTE_DTYPE)).all().item():
            raise RuntimeError('Shard tensor "{}" contains NaN or Inf: {}'.format(key, path))
    if payload.get('image_ids') != frame.iloc[start:end]['image_id'].tolist():
        raise RuntimeError('Feature shard image IDs are misaligned: {}'.format(path))
    return payload


def extract_shard(frame, start, end):
    window = frame.iloc[start:end]
    first_order = torch.empty((end - start,) + FIRST_ORDER_TAIL, dtype=FEATURE_STORAGE_DTYPE)
    gram = torch.empty((end - start,) + GRAM_TAIL, dtype=FEATURE_STORAGE_DTYPE)
    loader = DataLoader(
        ImageRows(window['path'].tolist()),
        batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=FEATURE_LOADER_WORKERS,
        pin_memory=True, persistent_workers=False,
    )
    cursor = 0
    for batch in loader:
        first_chunk, gram_chunk = extract_features(batch)
        rows = first_chunk.shape[0]
        first_order[cursor:cursor + rows] = first_chunk
        gram[cursor:cursor + rows] = gram_chunk
        cursor += rows
        del first_chunk, gram_chunk
    if cursor != end - start:
        raise RuntimeError('Shard extraction produced {} rows, expected {}.'.format(cursor, end - start))
    del loader
    return first_order, gram


def build_feature_cache(split, frame):
    """Resumable, atomic, signature-bound cache holding BOTH readouts from ONE forward pass."""
    row_signature = {'rows': int(len(frame)), 'image_ids_sha256': sha256_json(frame['image_id'].tolist())}
    if 'label' in frame.columns:
        row_signature['labels_sha256'] = sha256_json(frame['label'].astype(int).tolist())
    if 'fold' in frame.columns:
        row_signature['folds_sha256'] = sha256_json(frame['fold'].astype(int).tolist())
    identity = build_cache_identity(split, row_signature)
    cache_signature = sha256_json(identity)

    shard_dir = CACHE_DIR / split
    shard_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = ARTIFACT_DIR / 'cache_manifest_{}.json'.format(split)

    first_order = torch.empty((len(frame),) + FIRST_ORDER_TAIL, dtype=FEATURE_STORAGE_DTYPE)
    gram = torch.empty((len(frame),) + GRAM_TAIL, dtype=FEATURE_STORAGE_DTYPE)
    boundaries = [(s, min(s + FEATURE_SHARD_SIZE, len(frame))) for s in range(0, len(frame), FEATURE_SHARD_SIZE)]
    shard_records, reused_rows, written_rows, extraction_seconds = [], 0, 0, 0.0
    started_at = time.perf_counter()

    progress = tqdm(boundaries, desc='{} cache'.format(split))
    for start, end in progress:
        metadata = expected_shard_metadata(split, cache_signature, frame, start, end)
        shard_path = shard_dir / '{}_{:06d}_{:06d}.pt'.format(split, start, end)
        payload, reused = None, False
        if shard_path.is_file():
            try:
                payload = validate_shard(shard_path, metadata, frame, start, end)
                reused = True
            except RuntimeError:
                payload = None
        if payload is None:
            shard_started = time.perf_counter()
            shard_first, shard_gram = extract_shard(frame, start, end)
            extraction_seconds += time.perf_counter() - shard_started
            payload = {
                'metadata': metadata, 'first_order': shard_first, 'gram': shard_gram,
                'image_ids': frame.iloc[start:end]['image_id'].tolist(),
            }
            atomic_torch_save(payload, shard_path)
            written_rows += end - start
        else:
            reused_rows += end - start
        first_order[start:end] = payload['first_order']
        gram[start:end] = payload['gram']
        shard_records.append({'file': shard_path.name, 'start': start, 'end': end, 'reused': reused})
        del payload
        progress.set_postfix(reused=reused_rows, written=written_rows)

    if not torch.isfinite(first_order.to(POOLING_COMPUTE_DTYPE)).all().item():
        raise RuntimeError('The assembled first-order cache contains NaN or Inf.')

    manifest = {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'split': split,
        'complete': True,
        'rows': int(len(frame)),
        'cache_signature': cache_signature,
        'cache_identity': identity,
        'shard_count': len(boundaries),
        'rows_written_this_session': written_rows,
        'rows_reused_this_session': reused_rows,
        'first_order_bytes': int(first_order.numel() * first_order.element_size()),
        'gram_bytes': int(gram.numel() * gram.element_size()),
        'new_extraction_seconds': extraction_seconds,
        'new_extraction_images_per_second': (written_rows / extraction_seconds) if extraction_seconds > 0 else None,
        'elapsed_seconds_this_session': time.perf_counter() - started_at,
        'gpu_memory': gpu_memory_rows(),
        'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
        'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
        'shards': shard_records,
    }
    write_json_atomic(manifest, manifest_path)
    return first_order, gram, manifest


print('Feature cache helpers ready.')
print('First-order cache : {:.2f} GiB   Gram cache: {:.2f} GiB'.format(
    EXPECTED_TRAIN_ROWS * FIRST_ORDER_TAIL[0] * FIRST_ORDER_TAIL[1] * 2 / 2 ** 30,
    EXPECTED_TRAIN_ROWS * GRAM_TAIL[0] * GRAM_TAIL[1] * 2 / 2 ** 30,
))

In [ ]:
HEAD_DEVICE = 'cuda:0'


class CachedRows(Dataset):
    """Serves rows of the two cached tensors without copying them."""

    def __init__(self, first_order, gram, indices, labels=None):
        self.first_order = first_order
        self.gram = gram
        self.indices = [int(index) for index in indices]
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        label = int(self.labels[index]) if self.labels is not None else -1
        return self.first_order[index], self.gram[index], label, index


class FirstOrderHead(nn.Module):
    """The adopted V10 readout. hidden 512 is the champion; 640 is the capacity control."""

    def __init__(self, hidden_dim=HEAD_HIDDEN_DIM):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(FIRST_ORDER_DIM),
            nn.Linear(FIRST_ORDER_DIM, hidden_dim),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(hidden_dim, NUM_CLASSES),
        )

    def forward(self, first_order, gram):
        return self.network(first_order.flatten(1))


class GramTextureHead(nn.Module):
    """First-order champion readout plus a learned low-rank second-order branch.

    Second-order path, following Lin et al. 2015 (Bilinear CNN):
      G = W^T C W                     -> learned rank-R compression of the cached Gram
      g = triu(G), off-diagonal x sqrt(2)   -> keeps the Frobenius norm of the full matrix
      g = sign(g) * sqrt(|g|)         -> signed square root
      g = g / ||g||                   -> L2
    Both normalisation steps are mandatory in the bilinear-pooling literature; without them the
    features are badly scaled and train poorly.
    """

    def __init__(self):
        super().__init__()
        self.projection = nn.Parameter(torch.empty(GRAM_PROJECTION_DIM, GRAM_RANK))
        nn.init.orthogonal_(self.projection)
        triu_rows, triu_cols = torch.triu_indices(GRAM_RANK, GRAM_RANK)
        self.register_buffer('triu_rows', triu_rows)
        self.register_buffer('triu_cols', triu_cols)
        is_off_diagonal = triu_rows != triu_cols
        self.register_buffer(
            'triu_scale',
            torch.where(
                is_off_diagonal,
                torch.full((GRAM_TRIU_DIM,), math.sqrt(2.0)),
                torch.ones(GRAM_TRIU_DIM),
            ),
        )
        self.gram_branch = nn.Sequential(
            nn.LayerNorm(GRAM_TRIU_DIM),
            nn.Linear(GRAM_TRIU_DIM, GRAM_FUSION_DIM),
            nn.GELU(),
            nn.Dropout(GRAM_BRANCH_DROPOUT),
        )
        fused_dim = FIRST_ORDER_DIM + GRAM_FUSION_DIM
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, HEAD_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(HEAD_HIDDEN_DIM, NUM_CLASSES),
        )

    def forward(self, first_order, gram):
        projected = torch.matmul(gram, self.projection)             # [B, D, R]
        compact = torch.matmul(self.projection.t(), projected)      # [B, R, R] = W^T C W
        descriptor = compact[:, self.triu_rows, self.triu_cols] * self.triu_scale
        descriptor = torch.sign(descriptor) * torch.sqrt(torch.abs(descriptor) + 1e-12)
        descriptor = F.normalize(descriptor, dim=1, eps=1e-6)
        fused = torch.cat((first_order.flatten(1), self.gram_branch(descriptor)), dim=1)
        return self.classifier(fused)


def build_head(arm):
    if arm == CONTROL_ARM:
        return FirstOrderHead(HEAD_HIDDEN_DIM)
    if arm == CAPACITY_ARM:
        return FirstOrderHead(CAPACITY_HIDDEN_DIM)
    if arm == GRAM_ARM:
        return GramTextureHead()
    raise KeyError('Unknown arm: {}'.format(arm))


def softmax_numpy(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('Head probabilities contain NaN or Inf.')
    return probabilities


def predict_probabilities(model, first_order, gram, indices):
    index_list = [int(index) for index in indices]
    loader = DataLoader(CachedRows(first_order, gram, index_list), batch_size=PREDICT_BATCH_SIZE,
                        shuffle=False, num_workers=0, pin_memory=True)
    logits_parts, observed_indices = [], []
    model.eval()
    with torch.inference_mode():
        for batch_first, batch_gram, _batch_labels, batch_indices in loader:
            logits = model(
                batch_first.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True),
                batch_gram.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True),
            )
            if not torch.isfinite(logits).all().item():
                raise RuntimeError('Head logits contain NaN or Inf.')
            logits_parts.append(logits.float().cpu())
            observed_indices.extend(int(index) for index in batch_indices.tolist())
    if observed_indices != index_list:
        raise RuntimeError('Prediction order changed inside the DataLoader.')
    stacked = torch.cat(logits_parts).numpy().astype(np.float64)
    if stacked.shape != (len(index_list), NUM_CLASSES):
        raise RuntimeError('Head prediction matrix has shape {}.'.format(stacked.shape))
    return softmax_numpy(stacked)


def distribution_summary(values):
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        return {'count': 0, 'mean': None, 'std': None, 'q10': None, 'median': None, 'q90': None}
    return {
        'count': int(len(values)), 'mean': float(values.mean()), 'std': float(values.std()),
        'q10': float(np.quantile(values, 0.10)), 'median': float(np.quantile(values, 0.50)),
        'q90': float(np.quantile(values, 0.90)),
    }


def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(labels), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError('Metric input probabilities are incomplete or non-finite.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(NUM_CLASSES)))
    report = classification_report(labels, predictions, labels=list(range(NUM_CLASSES)),
                                   target_names=list(CLASS_NAMES), output_dict=True, zero_division=0)
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    confidence = top_two[:, 1]
    margin = top_two[:, 1] - top_two[:, 0]
    correct = predictions == labels
    recyclable_to_organic = int(matrix[0, 2])
    organic_to_recyclable = int(matrix[2, 0])
    return {
        'rows': int(len(labels)),
        'macro_f1': float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)), average='macro')),
        'accuracy': float(accuracy_score(labels, predictions)),
        'log_loss': float(log_loss(labels, probabilities, labels=list(range(NUM_CLASSES)))),
        'per_class': {
            name: {'precision': float(report[name]['precision']), 'recall': float(report[name]['recall']),
                   'f1': float(report[name]['f1-score']), 'support': int(report[name]['support'])}
            for name in CLASS_NAMES
        },
        'f1_recyclable': float(report['Recyclable']['f1-score']),
        'f1_electronic': float(report['Electronic']['f1-score']),
        'f1_organic': float(report['Organic']['f1-score']),
        'confusion_matrix': matrix.tolist(),
        'total_errors': int((~correct).sum()),
        'recyclable_to_organic': recyclable_to_organic,
        'organic_to_recyclable': organic_to_recyclable,
        'ro_errors': recyclable_to_organic + organic_to_recyclable,
        'electronic_errors': int(matrix[1].sum() - matrix[1, 1]),
        'high_confidence_errors': int(((~correct) & (confidence >= 0.90)).sum()),
        'confidence_summary': {'all': distribution_summary(confidence),
                               'correct': distribution_summary(confidence[correct]),
                               'errors': distribution_summary(confidence[~correct])},
        'margin_summary': {'all': distribution_summary(margin),
                           'correct': distribution_summary(margin[correct]),
                           'errors': distribution_summary(margin[~correct])},
    }


def predictions_frame(indices, probabilities, arm, seed, validation_fold):
    indices = np.asarray(indices, dtype=np.int64)
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    predictions = probabilities.argmax(axis=1)
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    return pd.DataFrame({
        'row_index': indices,
        'image_id': train_df.iloc[indices]['image_id'].to_numpy(),
        'label': labels, 'prediction': predictions,
        'prob_0': probabilities[:, 0], 'prob_1': probabilities[:, 1], 'prob_2': probabilities[:, 2],
        'confidence': top_two[:, 1], 'top_two_margin': top_two[:, 1] - top_two[:, 0],
        'correct': labels == predictions,
        'fold': validation_fold, 'arm': arm, 'seed': seed,
    })


def class_weights_from(label_values):
    class_counts = np.bincount(label_values, minlength=NUM_CLASSES)
    if (class_counts <= 0).any():
        raise RuntimeError('A training partition is missing a class: {}.'.format(class_counts.tolist()))
    weights = class_counts.sum() / (NUM_CLASSES * class_counts)
    return class_counts, torch.tensor(weights, dtype=torch.float32, device=HEAD_DEVICE)


def run_training_epochs(model, first_order, gram, train_indices, epochs, seed,
                        criterion, optimizer, scheduler, on_epoch_end):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        CachedRows(first_order, gram, train_indices, cached_labels),
        batch_size=HEAD_BATCH_SIZE, shuffle=True, generator=generator,
        num_workers=0, pin_memory=True, drop_last=False,
    )
    for epoch in range(1, epochs + 1):
        epoch_started = time.perf_counter()
        model.train()
        loss_sum, seen = 0.0, 0
        learning_rate = optimizer.param_groups[0]['lr']
        for batch_first, batch_gram, batch_labels, _batch_indices in train_loader:
            batch_first = batch_first.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True)
            batch_gram = batch_gram.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True)
            batch_labels = batch_labels.to(device=HEAD_DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_first, batch_gram), batch_labels)
            if not torch.isfinite(loss).item():
                raise RuntimeError('Non-finite loss at epoch {}.'.format(epoch))
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            if not torch.isfinite(gradient_norm).item():
                raise RuntimeError('Non-finite head gradient norm at epoch {}.'.format(epoch))
            optimizer.step()
            loss_sum += loss.item() * len(batch_labels)
            seen += len(batch_labels)
        scheduler.step()
        if on_epoch_end(epoch, loss_sum / seen, learning_rate, time.perf_counter() - epoch_started):
            break
    del train_loader


def train_fold_head(arm, first_order, gram, validation_fold, seed, fold_values=None, persist=True):
    """One arm, one fold, one seed, the locked head protocol. Every arm is trained from scratch."""
    if fold_values is None:
        fold_values = cached_folds.numpy()
    train_indices = np.flatnonzero(fold_values != validation_fold)
    valid_indices = np.flatnonzero(fold_values == validation_fold)
    if len(train_indices) == 0 or len(valid_indices) == 0:
        raise RuntimeError('Empty train or validation partition for fold {}.'.format(validation_fold))

    train_label_values = cached_labels[torch.from_numpy(train_indices)].numpy()
    valid_labels = cached_labels[torch.from_numpy(valid_indices)].numpy()
    class_counts, class_weights = class_weights_from(train_label_values)

    seed_everything(seed)
    model = build_head(arm).to(HEAD_DEVICE)
    parameter_count = sum(p.numel() for p in model.parameters())
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    state = {'best_macro_f1': -1.0, 'best_log_loss': float('inf'), 'best_epoch': None,
             'best_state': None, 'stale': 0}
    history = []
    training_started = time.perf_counter()

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        valid_probabilities = predict_probabilities(model, first_order, gram, valid_indices)
        valid_predictions = valid_probabilities.argmax(axis=1)
        valid_macro_f1 = float(f1_score(valid_labels, valid_predictions,
                                        labels=list(range(NUM_CLASSES)), average='macro'))
        valid_log_loss = float(log_loss(valid_labels, valid_probabilities, labels=list(range(NUM_CLASSES))))
        history.append({'arm': arm, 'validation_fold': validation_fold, 'seed': seed, 'epoch': epoch,
                        'train_loss': train_loss, 'valid_macro_f1': valid_macro_f1,
                        'valid_log_loss': valid_log_loss, 'learning_rate': learning_rate,
                        'epoch_seconds': epoch_seconds})
        improved = valid_macro_f1 > state['best_macro_f1'] + 1e-8 or (
            abs(valid_macro_f1 - state['best_macro_f1']) <= 1e-8 and valid_log_loss < state['best_log_loss'])
        if improved:
            state.update({'best_macro_f1': valid_macro_f1, 'best_log_loss': valid_log_loss,
                          'best_epoch': epoch, 'stale': 0,
                          'best_state': {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}})
        else:
            state['stale'] += 1
        return state['stale'] >= EARLY_STOPPING_PATIENCE

    run_training_epochs(model, first_order, gram, train_indices, MAX_EPOCHS, seed,
                        criterion, optimizer, scheduler, on_epoch_end)
    if state['best_state'] is None:
        raise RuntimeError('Head training produced no valid checkpoint.')
    training_seconds = time.perf_counter() - training_started

    model.load_state_dict(state['best_state'], strict=True)
    final_probabilities = predict_probabilities(model, first_order, gram, valid_indices)
    metrics = metric_bundle(valid_labels, final_probabilities)
    metrics.update({
        'arm': arm, 'validation_fold': validation_fold, 'seed': seed,
        'best_epoch': state['best_epoch'],
        'hit_epoch_cap': bool(state['best_epoch'] == MAX_EPOCHS),
        'parameter_count': parameter_count, 'training_seconds': training_seconds,
        'train_class_counts': class_counts.tolist(),
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    })

    prediction_table = predictions_frame(valid_indices, final_probabilities, arm, seed, validation_fold)
    if persist:
        run_stem = '{}_fold{}_seed{}'.format(arm, validation_fold, seed)
        write_csv_atomic(prediction_table, HEADS_DIR / (run_stem + '_predictions.csv'))
        write_csv_atomic(pd.DataFrame(history), HEADS_DIR / (run_stem + '_history.csv'))
        write_json_atomic(metrics, HEADS_DIR / (run_stem + '_metrics.json'))

    del model, criterion, optimizer, scheduler
    state['best_state'] = None
    release_cuda_cache()
    return metrics, prediction_table


def run_five_fold_oof(arm, first_order, gram, seed, fold_values=None, persist=True, label=''):
    """Full five-fold OOF over all rows. No fold ensembling: every row is predicted exactly once."""
    per_fold, frames = [], []
    for fold in range(N_FOLDS):
        metrics, frame = train_fold_head(arm, first_order, gram, fold, seed,
                                         fold_values=fold_values, persist=persist)
        per_fold.append(metrics)
        frames.append(frame)
        print('  [{}{}] seed={} fold={} macro_f1={:.10f} err={} R/O={} elecF1={:.6f} epoch={}{}'.format(
            arm, label, seed, fold, metrics['macro_f1'], metrics['total_errors'], metrics['ro_errors'],
            metrics['f1_electronic'], metrics['best_epoch'], ' CAP' if metrics['hit_epoch_cap'] else ''))
    pooled = pd.concat(frames, ignore_index=True).sort_values('row_index').reset_index(drop=True)
    if pooled['row_index'].tolist() != list(range(len(train_df))):
        raise RuntimeError('OOF predictions are incomplete or misaligned.')
    oof = metric_bundle(pooled['label'].to_numpy(dtype=np.int64),
                        pooled[['prob_0', 'prob_1', 'prob_2']].to_numpy(dtype=np.float64))
    oof.update({
        'arm': arm, 'seed': seed,
        'per_fold_macro_f1': {str(m['validation_fold']): m['macro_f1'] for m in per_fold},
        'best_epochs': {str(m['validation_fold']): m['best_epoch'] for m in per_fold},
        'folds_hitting_epoch_cap': int(sum(1 for m in per_fold if m['hit_epoch_cap'])),
        'parameter_count': per_fold[0]['parameter_count'],
    })
    return per_fold, oof, pooled


ARM_PARAMETER_COUNTS = {}
for arm in ARM_ORDER:
    probe = build_head(arm)
    ARM_PARAMETER_COUNTS[arm] = sum(p.numel() for p in probe.parameters())
    with torch.no_grad():
        probe_logits = probe(torch.zeros(2, *FIRST_ORDER_TAIL), torch.zeros(2, *GRAM_TAIL))
    if tuple(probe_logits.shape) != (2, NUM_CLASSES):
        raise RuntimeError('Arm {} produced logits with shape {}.'.format(arm, tuple(probe_logits.shape)))
    del probe, probe_logits

if ARM_PARAMETER_COUNTS[CONTROL_ARM] != V10_REFERENCE['head_parameter_count']:
    raise RuntimeError(
        'Control head parameter count {:,} does not match the adopted V10 head ({:,}); the control is not '
        'the champion head.'.format(ARM_PARAMETER_COUNTS[CONTROL_ARM], V10_REFERENCE['head_parameter_count'])
    )

print('Head parameter counts:')
for arm in ARM_ORDER:
    print('  {:<18} {:>12,}'.format(arm, ARM_PARAMETER_COUNTS[arm]))
print('  V10 champion       {:>12,}  (control must match exactly -- it does)'.format(
    V10_REFERENCE['head_parameter_count']))
print('capacity_control exists so a gram_texture win cannot be confused with "more parameters".')

In [ ]:
print('=' * 78)
print('EXTRACTION  (one forward pass, both readouts)   [t+{:.0f} min]'.format(elapsed_minutes()))
print('=' * 78)

TRAIN_FIRST_ORDER, TRAIN_GRAM, TRAIN_CACHE_MANIFEST = build_feature_cache('train', train_df)
release_cuda_cache()
print('Cache ready: {} rows | first-order {:.2f} GiB | gram {:.2f} GiB | written {} reused {}'.format(
    TRAIN_CACHE_MANIFEST['rows'],
    TRAIN_CACHE_MANIFEST['first_order_bytes'] / 2 ** 30,
    TRAIN_CACHE_MANIFEST['gram_bytes'] / 2 ** 30,
    TRAIN_CACHE_MANIFEST['rows_written_this_session'],
    TRAIN_CACHE_MANIFEST['rows_reused_this_session'],
))
print('Cache signature:', TRAIN_CACHE_MANIFEST['cache_signature'])

print()
print('=' * 78)
print('FIVE-FOLD OOF, SEED {}   [t+{:.0f} min]'.format(SEED, elapsed_minutes()))
print('=' * 78)

PER_FOLD_METRICS, OOF_METRICS, OOF_PREDICTIONS = {}, {}, {}
for arm in ARM_ORDER:
    role = 'DIAGNOSTIC' if arm in DIAGNOSTIC_ARMS else ('control' if arm == CONTROL_ARM else 'CANDIDATE')
    print()
    print('--- {} ({}) ---'.format(arm, role))
    per_fold, oof, pooled = run_five_fold_oof(arm, TRAIN_FIRST_ORDER, TRAIN_GRAM, SEED, persist=True)
    PER_FOLD_METRICS[arm] = per_fold
    OOF_METRICS[arm] = oof
    OOF_PREDICTIONS[arm] = pooled

    write_csv_atomic(pooled, ARTIFACT_DIR / 'oof_predictions_{}.csv'.format(arm))
    write_npy_atomic(pooled[['prob_0', 'prob_1', 'prob_2']].to_numpy(dtype=np.float32),
                     ARTIFACT_DIR / 'oof_probabilities_{}.npy'.format(arm))
    write_json_atomic(oof, ARTIFACT_DIR / 'oof_metrics_{}.json'.format(arm))

    print('  FULL OOF  macro_f1={:.10f}  errors={}  R/O={} (R->O {}, O->R {})'.format(
        oof['macro_f1'], oof['total_errors'], oof['ro_errors'],
        oof['recyclable_to_organic'], oof['organic_to_recyclable']))
    print('            Electronic err={} F1={:.10f} | R F1={:.10f} | O F1={:.10f}'.format(
        oof['electronic_errors'], oof['f1_electronic'], oof['f1_recyclable'], oof['f1_organic']))
    print('            log_loss={:.10f}  high-confidence errors={}  folds at epoch cap={}/5'.format(
        oof['log_loss'], oof['high_confidence_errors'], oof['folds_hitting_epoch_cap']))

per_fold_table = pd.DataFrame([
    {
        'arm': m['arm'], 'seed': m['seed'], 'validation_fold': m['validation_fold'],
        'best_epoch': m['best_epoch'], 'hit_epoch_cap': m['hit_epoch_cap'],
        'macro_f1': m['macro_f1'], 'accuracy': m['accuracy'], 'log_loss': m['log_loss'],
        'total_errors': m['total_errors'], 'recyclable_to_organic': m['recyclable_to_organic'],
        'organic_to_recyclable': m['organic_to_recyclable'], 'ro_errors': m['ro_errors'],
        'electronic_errors': m['electronic_errors'], 'f1_electronic': m['f1_electronic'],
        'parameter_count': m['parameter_count'], 'training_seconds': m['training_seconds'],
    }
    for arm in ARM_ORDER for m in PER_FOLD_METRICS[arm]
]).sort_values(['arm', 'validation_fold']).reset_index(drop=True)
write_csv_atomic(per_fold_table, ARTIFACT_DIR / 'per_fold_seed42.csv')

print()
print('=' * 78)
print('PER-FOLD TABLE (seed {})'.format(SEED))
print('=' * 78)
display(per_fold_table[['arm', 'validation_fold', 'best_epoch', 'hit_epoch_cap', 'macro_f1',
                        'total_errors', 'ro_errors', 'electronic_errors', 'f1_electronic']])
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
# ---------------------------------------------------------------------------
# PIPELINE INTEGRITY: does the in-session control reproduce the recorded champion?
# Checked BEFORE the adoption gate. V13's control hit macro_f1_delta = 0.0 exactly,
# so anything materially different here means something changed in the environment.
# ---------------------------------------------------------------------------
control_oof = OOF_METRICS[CONTROL_ARM]
control_macro_f1_delta = control_oof['macro_f1'] - V10_REFERENCE['oof_macro_f1']

CONTROL_REPRODUCTION = {
    'v10_recorded_macro_f1': V10_REFERENCE['oof_macro_f1'],
    'v14_in_session_control_macro_f1': control_oof['macro_f1'],
    'macro_f1_delta': control_macro_f1_delta,
    'tolerance': CONTROL_REPRODUCTION_TOLERANCE,
    'within_tolerance': bool(abs(control_macro_f1_delta) <= CONTROL_REPRODUCTION_TOLERANCE),
    'exactly_reproduced': bool(control_macro_f1_delta == 0.0),
    'v10_recorded_total_errors': V10_REFERENCE['oof_total_errors'],
    'v14_control_total_errors': control_oof['total_errors'],
    'total_error_delta': control_oof['total_errors'] - V10_REFERENCE['oof_total_errors'],
    'v10_recorded_ro_errors': V10_REFERENCE['oof_ro_errors'],
    'v14_control_ro_errors': control_oof['ro_errors'],
    'ro_error_delta': control_oof['ro_errors'] - V10_REFERENCE['oof_ro_errors'],
    'model_revision_matches_v10': MODEL_REVISION_MATCHES_V10,
    'v13_precedent': 'V13 reproduced the champion exactly: macro_f1_delta 0.0, 287 errors, 242 R/O',
}
PIPELINE_INTEGRITY_OK = CONTROL_REPRODUCTION['within_tolerance']
write_json_atomic(CONTROL_REPRODUCTION, ARTIFACT_DIR / 'control_reproduction.json')

print('=' * 78)
print('PIPELINE INTEGRITY CHECK')
print('=' * 78)
print('V10 recorded OOF macro F1 : {:.10f}  (errors {}, R/O {})'.format(
    V10_REFERENCE['oof_macro_f1'], V10_REFERENCE['oof_total_errors'], V10_REFERENCE['oof_ro_errors']))
print('V14 in-session control    : {:.10f}  (errors {}, R/O {})'.format(
    control_oof['macro_f1'], control_oof['total_errors'], control_oof['ro_errors']))
print('delta                     : {:+.10f}  (tolerance +/-{:.4f})'.format(
    control_macro_f1_delta, CONTROL_REPRODUCTION_TOLERANCE))
print('exactly reproduced        :', CONTROL_REPRODUCTION['exactly_reproduced'], '(V13 achieved this)')
print('PIPELINE INTEGRITY        :', 'PASS' if PIPELINE_INTEGRITY_OK else 'FAIL')
if not PIPELINE_INTEGRITY_OK:
    print()
    print('!' * 78)
    print('!! The control did not reproduce the champion. Treat every number below as       !!')
    print('!! UNINTERPRETABLE. Adoption is refused automatically and the submission is      !!')
    print('!! built from the control recipe, which is what V10 already used.                !!')
    print('!' * 78)

# ---------------------------------------------------------------------------
# SEED STABILITY: confirmation only, never selection. Seed 42 alone drives the decision.
# Only the two adoption-relevant arms are repeated; capacity_control stays a seed-42 diagnostic.
# ---------------------------------------------------------------------------
print()
print('=' * 78)
print('SEED STABILITY (confirmation, never selection)   [t+{:.0f} min]'.format(elapsed_minutes()))
print('=' * 78)

STABILITY_ARMS = (CONTROL_ARM, GRAM_ARM)
SEED_OOF = {arm: {SEED: OOF_METRICS[arm]} for arm in STABILITY_ARMS}
for stability_seed in STABILITY_SEEDS:
    for arm in STABILITY_ARMS:
        _, oof, _ = run_five_fold_oof(arm, TRAIN_FIRST_ORDER, TRAIN_GRAM, stability_seed, persist=False)
        SEED_OOF[arm][stability_seed] = oof

ALL_SEEDS = (SEED,) + tuple(STABILITY_SEEDS)
seed_rows = []
for seed_value in ALL_SEEDS:
    control_seed_oof = SEED_OOF[CONTROL_ARM][seed_value]
    gram_seed_oof = SEED_OOF[GRAM_ARM][seed_value]
    seed_rows.append({
        'seed': seed_value,
        'control_macro_f1': control_seed_oof['macro_f1'],
        'gram_macro_f1': gram_seed_oof['macro_f1'],
        'macro_f1_gain': gram_seed_oof['macro_f1'] - control_seed_oof['macro_f1'],
        'control_total_errors': control_seed_oof['total_errors'],
        'gram_total_errors': gram_seed_oof['total_errors'],
        'control_ro_errors': control_seed_oof['ro_errors'],
        'gram_ro_errors': gram_seed_oof['ro_errors'],
        'control_f1_electronic': control_seed_oof['f1_electronic'],
        'gram_f1_electronic': gram_seed_oof['f1_electronic'],
        'v13_control_macro_f1': V13_CONTROL_SEED_MACRO_F1.get(str(seed_value)),
        'v13_control_total_errors': V13_CONTROL_SEED_TOTAL_ERRORS.get(str(seed_value)),
    })
seed_stability_table = pd.DataFrame(seed_rows)
write_csv_atomic(seed_stability_table, ARTIFACT_DIR / 'seed_stability.csv')

SEED_GAINS = {str(row['seed']): float(row['macro_f1_gain']) for row in seed_rows}
gain_signs = {1 if v > 0 else (-1 if v < 0 else 0) for v in SEED_GAINS.values()}
SEED_SIGN_CONSISTENT = len(gain_signs) == 1 and 0 not in gain_signs
CONTROL_SEED_SPREAD = float(np.std([row['control_macro_f1'] for row in seed_rows]))

print()
display(seed_stability_table)
print('Gain by seed        :', {k: round(v, 8) for k, v in SEED_GAINS.items()})
print('Sign consistent     :', SEED_SIGN_CONSISTENT)
print('Control seed std    : {:.8f}  (V13 measured {:.8f} on the same arm)'.format(
    CONTROL_SEED_SPREAD, V13_CONTROL_SEED_SPREAD))
print('  -> Read the seed-42 gain against 2x this number. The +0.0003 gate is only ~1.4 sigma,')
print('     so a gain smaller than the spread is noise no matter what the gate says.')
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
# ---------------------------------------------------------------------------
# DIAGNOSTIC ONLY: duplicate-grouped validation (CONTEXT section 7.3).
#
# This NEVER feeds the adoption gate. Grouped folds are not protocol-matched to V10, so the
# grouped number is not comparable to 0.9901662323927614 and a drop here is a measurement
# correction, not a regression. Everything is wrapped so a diagnostic failure can never
# destroy a run that already spent its extraction budget.
# ---------------------------------------------------------------------------
GROUPED_DIAGNOSTIC = {'attempted': True, 'succeeded': False}
try:
    print('=' * 78)
    print('GROUPED-VALIDATION DIAGNOSTIC   [t+{:.0f} min]'.format(elapsed_minutes()))
    print('=' * 78)

    similarity_device = HEAD_DEVICE
    normalised = F.normalize(
        TRAIN_FIRST_ORDER.flatten(1).to(device=similarity_device, dtype=torch.float16), dim=1
    )
    row_count = normalised.shape[0]

    def connected_components(threshold, chunk_size=2048):
        parent = list(range(row_count))

        def find(node):
            while parent[node] != node:
                parent[node] = parent[parent[node]]
                node = parent[node]
            return node

        def union(left, right):
            left_root, right_root = find(left), find(right)
            if left_root != right_root:
                parent[max(left_root, right_root)] = min(left_root, right_root)

        for start in range(0, row_count, chunk_size):
            end = min(start + chunk_size, row_count)
            block = torch.matmul(normalised[start:end], normalised.t()).float()
            # Keep only strictly-upper pairs so each pair is visited once.
            column_index = torch.arange(row_count, device=block.device).unsqueeze(0)
            row_index = torch.arange(start, end, device=block.device).unsqueeze(1)
            hits = (block >= threshold) & (column_index > row_index)
            pairs = torch.nonzero(hits, as_tuple=False)
            for left, right in pairs.cpu().tolist():
                union(start + left, right)
            del block, hits, pairs
        groups = np.array([find(node) for node in range(row_count)], dtype=np.int64)
        _, groups = np.unique(groups, return_inverse=True)
        return groups

    threshold_rows = []
    grouping_by_threshold = {}
    for threshold in GROUP_SIMILARITY_THRESHOLDS:
        groups = connected_components(threshold)
        sizes = np.bincount(groups)
        multi = sizes[sizes > 1]
        threshold_rows.append({
            'threshold': threshold,
            'group_count_multi_member': int(len(multi)),
            'files_in_multi_member_groups': int(multi.sum()),
            'largest_group': int(sizes.max()),
            'distinct_groups_total': int(len(sizes)),
        })
        grouping_by_threshold[threshold] = groups
        print('  tau={:.2f} -> {} multi-member groups covering {} files (largest {})'.format(
            threshold, len(multi), int(multi.sum()), int(sizes.max())))

    del normalised
    release_cuda_cache()

    threshold_table = pd.DataFrame(threshold_rows)
    write_csv_atomic(threshold_table, ARTIFACT_DIR / 'near_duplicate_thresholds.csv')
    print()
    print('V9 reference: {} groups covering {} files.'.format(
        V9_REFERENCE_GROUP_COUNT, V9_REFERENCE_GROUPED_FILES))
    print('V9 used a different detector, so exact agreement is not expected. The threshold whose file')
    print('count is closest to the V9 scale is used for the grouped run, purely to anchor the scale.')

    chosen = min(threshold_rows, key=lambda row: abs(row['files_in_multi_member_groups']
                                                     - V9_REFERENCE_GROUPED_FILES))
    chosen_threshold = chosen['threshold']
    chosen_groups = grouping_by_threshold[chosen_threshold]
    print('Chosen tau  : {:.2f} ({} files in multi-member groups)'.format(
        chosen_threshold, chosen['files_in_multi_member_groups']))

    grouped_splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    grouped_fold_values = np.full(row_count, -1, dtype=np.int64)
    for fold_index, (_, valid_positions) in enumerate(
        grouped_splitter.split(train_df, train_df['label'], groups=chosen_groups)
    ):
        grouped_fold_values[valid_positions] = fold_index
    if (grouped_fold_values < 0).any():
        raise RuntimeError('StratifiedGroupKFold left rows unassigned.')

    crossing = 0
    for group_id in np.unique(chosen_groups):
        member_folds = np.unique(grouped_fold_values[chosen_groups == group_id])
        if len(member_folds) > 1:
            crossing += 1
    print('Groups still crossing folds after grouping: {} (should be 0)'.format(crossing))

    print()
    grouped_oof = {}
    for arm in (CONTROL_ARM, GRAM_ARM):
        _, oof, _ = run_five_fold_oof(arm, TRAIN_FIRST_ORDER, TRAIN_GRAM, SEED,
                                      fold_values=grouped_fold_values, persist=False, label=' grouped')
        grouped_oof[arm] = oof

    GROUPED_DIAGNOSTIC = {
        'attempted': True,
        'succeeded': True,
        'is_a_gate': False,
        'threshold_sweep': threshold_rows,
        'chosen_threshold': chosen_threshold,
        'files_in_multi_member_groups': chosen['files_in_multi_member_groups'],
        'group_count_multi_member': chosen['group_count_multi_member'],
        'groups_crossing_folds_after_grouping': int(crossing),
        'v9_reference_group_count': V9_REFERENCE_GROUP_COUNT,
        'v9_reference_grouped_files': V9_REFERENCE_GROUPED_FILES,
        'grouped_macro_f1': {arm: grouped_oof[arm]['macro_f1'] for arm in grouped_oof},
        'grouped_total_errors': {arm: grouped_oof[arm]['total_errors'] for arm in grouped_oof},
        'grouped_ro_errors': {arm: grouped_oof[arm]['ro_errors'] for arm in grouped_oof},
        'grouped_gain': grouped_oof[GRAM_ARM]['macro_f1'] - grouped_oof[CONTROL_ARM]['macro_f1'],
        'stratified_macro_f1': {arm: OOF_METRICS[arm]['macro_f1'] for arm in (CONTROL_ARM, GRAM_ARM)},
        'interpretation': (
            'A lower grouped number than the stratified number is the honest cost of near-duplicate '
            'leakage, not a regression. What matters is whether the control-to-candidate gain keeps '
            'the same sign under both fold schemes.'
        ),
    }
    write_json_atomic(GROUPED_DIAGNOSTIC, ARTIFACT_DIR / 'grouped_validation_diagnostic.json')

    print()
    print('  grouped   control macro_f1={:.10f} errors={} R/O={}'.format(
        grouped_oof[CONTROL_ARM]['macro_f1'], grouped_oof[CONTROL_ARM]['total_errors'],
        grouped_oof[CONTROL_ARM]['ro_errors']))
    print('  grouped   gram    macro_f1={:.10f} errors={} R/O={}'.format(
        grouped_oof[GRAM_ARM]['macro_f1'], grouped_oof[GRAM_ARM]['total_errors'],
        grouped_oof[GRAM_ARM]['ro_errors']))
    print('  grouped   gain   = {:+.10f}'.format(GROUPED_DIAGNOSTIC['grouped_gain']))
    print('  stratified gain  = {:+.10f}'.format(
        OOF_METRICS[GRAM_ARM]['macro_f1'] - OOF_METRICS[CONTROL_ARM]['macro_f1']))
    print('  leakage cost on the control = {:+.10f} (grouped minus stratified)'.format(
        grouped_oof[CONTROL_ARM]['macro_f1'] - OOF_METRICS[CONTROL_ARM]['macro_f1']))
except Exception as exc:
    GROUPED_DIAGNOSTIC = {'attempted': True, 'succeeded': False, 'is_a_gate': False,
                          'error_type': type(exc).__name__, 'error': str(exc)}
    write_json_atomic(GROUPED_DIAGNOSTIC, ARTIFACT_DIR / 'grouped_validation_diagnostic.json')
    release_cuda_cache()
    print()
    print('Grouped-validation diagnostic FAILED: {}: {}'.format(type(exc).__name__, exc))
    print('This is diagnostic-only and deliberately non-fatal. The adoption gate is unaffected')
    print('and the run continues.')

print()
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
control_oof = OOF_METRICS[CONTROL_ARM]
gram_oof = OOF_METRICS[GRAM_ARM]
capacity_oof = OOF_METRICS[CAPACITY_ARM]

OOF_MACRO_F1_GAIN = gram_oof['macro_f1'] - control_oof['macro_f1']
CAPACITY_GAIN = capacity_oof['macro_f1'] - control_oof['macro_f1']
ELECTRONIC_F1_DROP = control_oof['f1_electronic'] - gram_oof['f1_electronic']

fold_macro_f1 = {
    arm: {int(m['validation_fold']): float(m['macro_f1']) for m in PER_FOLD_METRICS[arm]}
    for arm in ARM_ORDER
}
ASSESSMENT_FOLD_GAINS = {
    str(fold): fold_macro_f1[GRAM_ARM][fold] - fold_macro_f1[CONTROL_ARM][fold]
    for fold in ASSESSMENT_FOLDS
}
DEVELOPMENT_FOLD_GAIN = (fold_macro_f1[GRAM_ARM][DEVELOPMENT_FOLD]
                         - fold_macro_f1[CONTROL_ARM][DEVELOPMENT_FOLD])
assessment_gains = list(ASSESSMENT_FOLD_GAINS.values())
FOLDS_WON = sum(1 for gain in assessment_gains if gain > 0)
positive_gain_total = sum(gain for gain in assessment_gains if gain > 0)
GAIN_CONCENTRATION = (max(assessment_gains) / positive_gain_total) if positive_gain_total > 0 else 1.0

ADOPTION_GATE_CHECKS = {
    'pipeline_integrity_control_reproduces_v10': bool(PIPELINE_INTEGRITY_OK),
    'oof_macro_f1_gain_at_least_threshold': bool(OOF_MACRO_F1_GAIN >= ADOPTION_MIN_OOF_MACRO_F1_GAIN),
    'oof_total_errors_decrease': bool(gram_oof['total_errors'] < control_oof['total_errors']),
    'oof_ro_errors_decrease': bool(gram_oof['ro_errors'] < control_oof['ro_errors']),
    'wins_at_least_three_of_four_assessment_folds': bool(FOLDS_WON >= ADOPTION_MIN_ASSESSMENT_FOLDS_WON),
    'electronic_f1_stable': bool(ELECTRONIC_F1_DROP <= ADOPTION_MAX_ELECTRONIC_F1_DROP),
    'improvement_not_concentrated_in_one_fold': bool(
        GAIN_CONCENTRATION <= ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE),
    'gain_sign_consistent_across_seeds': bool(SEED_SIGN_CONSISTENT),
    'no_test_access_during_selection': bool(
        not TEST_ACCESS_STATE['test_images_accessed']
        and not TEST_ACCESS_STATE['manual_test_labels_accessed']),
}
GRAM_ADOPTED = all(ADOPTION_GATE_CHECKS.values())
SELECTED_ARM = GRAM_ARM if GRAM_ADOPTED else CONTROL_ARM
SELECTION_REASON = (
    'gram_texture cleared every locked gate; the second-order branch was the only change'
    if GRAM_ADOPTED
    else 'gram_texture failed at least one locked gate; the V10 control recipe stands as champion'
)

GAIN_EXCEEDS_SEED_NOISE = bool(abs(OOF_MACRO_F1_GAIN) > 2.0 * CONTROL_SEED_SPREAD)
CAPACITY_EXPLAINS_GAIN = bool(CAPACITY_GAIN >= 0.5 * OOF_MACRO_F1_GAIN and OOF_MACRO_F1_GAIN > 0)
STRONG_EVIDENCE = bool(OOF_MACRO_F1_GAIN >= STRONG_EVIDENCE_MACRO_F1_GAIN)

SELECTION_DECISION = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'selected_arm': SELECTED_ARM,
    'selection_reason': SELECTION_REASON,
    'selection_basis': 'train_only_cross_validation',
    'gram_adopted': GRAM_ADOPTED,
    'adoption_gate_checks': ADOPTION_GATE_CHECKS,
    'oof_macro_f1_gain': OOF_MACRO_F1_GAIN,
    'electronic_f1_drop': ELECTRONIC_F1_DROP,
    'development_fold_gain': DEVELOPMENT_FOLD_GAIN,
    'assessment_fold_gains': ASSESSMENT_FOLD_GAINS,
    'assessment_folds_won': FOLDS_WON,
    'largest_single_fold_gain_share': GAIN_CONCENTRATION,
    'seed_gains': SEED_GAINS,
    'control_seed_spread': CONTROL_SEED_SPREAD,
    'gain_exceeds_twice_seed_noise': GAIN_EXCEEDS_SEED_NOISE,
    'strong_evidence_threshold_met': STRONG_EVIDENCE,
    'capacity_control_gain': CAPACITY_GAIN,
    'capacity_explains_at_least_half_the_gain': CAPACITY_EXPLAINS_GAIN,
    'arm_parameter_counts': ARM_PARAMETER_COUNTS,
    'folds_hitting_epoch_cap': {arm: OOF_METRICS[arm]['folds_hitting_epoch_cap'] for arm in ARM_ORDER},
    'out_of_fold_metrics': {arm: OOF_METRICS[arm] for arm in ARM_ORDER},
    'control_reproduction': CONTROL_REPRODUCTION,
    'per_fold_macro_f1': fold_macro_f1,
    'grouped_validation_diagnostic': GROUPED_DIAGNOSTIC,
    'blending_used': False,
    'fold_ensembling_used': False,
    'seed_ensembling_used': False,
    'threshold_tuning_used': False,
    'warm_start_used': False,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'preflight_signature': PREFLIGHT_SIGNATURE,
    'train_cache_signature': TRAIN_CACHE_MANIFEST['cache_signature'],
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
SELECTION_DECISION_SIGNATURE = sha256_json(SELECTION_DECISION)
write_json_atomic(
    {'selection_decision': SELECTION_DECISION, 'selection_decision_signature': SELECTION_DECISION_SIGNATURE},
    SELECTION_DECISION_PATH,
)

print('=' * 78)
print('ADOPTION GATE   (locked before any V14 number existed)')
print('=' * 78)
print()
header = '{:<26} {:>16} {:>16} {:>16}'
print(header.format('metric', 'control', 'gram_texture', 'capacity(diag)'))
print('-' * 78)
print(header.format('OOF macro F1', '{:.10f}'.format(control_oof['macro_f1']),
                    '{:.10f}'.format(gram_oof['macro_f1']), '{:.10f}'.format(capacity_oof['macro_f1'])))
for key, name in (('total_errors', 'OOF total errors'), ('ro_errors', 'OOF R/O errors'),
                  ('recyclable_to_organic', '  R->O'), ('organic_to_recyclable', '  O->R'),
                  ('electronic_errors', 'OOF Electronic errors'),
                  ('high_confidence_errors', 'high-confidence errors')):
    print(header.format(name, control_oof[key], gram_oof[key], capacity_oof[key]))
for key, name in (('f1_electronic', 'Electronic F1'), ('f1_recyclable', 'Recyclable F1'),
                  ('f1_organic', 'Organic F1'), ('log_loss', 'log loss')):
    print(header.format(name, '{:.10f}'.format(control_oof[key]), '{:.10f}'.format(gram_oof[key]),
                        '{:.10f}'.format(capacity_oof[key])))
print(header.format('head parameters', '{:,}'.format(ARM_PARAMETER_COUNTS[CONTROL_ARM]),
                    '{:,}'.format(ARM_PARAMETER_COUNTS[GRAM_ARM]),
                    '{:,}'.format(ARM_PARAMETER_COUNTS[CAPACITY_ARM])))
print(header.format('folds at epoch cap', control_oof['folds_hitting_epoch_cap'],
                    gram_oof['folds_hitting_epoch_cap'], capacity_oof['folds_hitting_epoch_cap']))
print()
print('OOF macro F1 gain     : {:+.10f}   (gate {:+.4f})'.format(
    OOF_MACRO_F1_GAIN, ADOPTION_MIN_OOF_MACRO_F1_GAIN))
print('Fold 0 gain           : {:+.10f}'.format(DEVELOPMENT_FOLD_GAIN))
print('Assessment fold gains : {}'.format({k: round(v, 8) for k, v in ASSESSMENT_FOLD_GAINS.items()}))
print('Assessment folds won  : {} of 4   (gate >= {})'.format(FOLDS_WON, ADOPTION_MIN_ASSESSMENT_FOLDS_WON))
print('Largest fold share    : {:.4f}   (gate <= {:.2f})'.format(
    GAIN_CONCENTRATION, ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE))
print('Electronic F1 drop    : {:+.10f}   (gate <= {:.4f})'.format(
    ELECTRONIC_F1_DROP, ADOPTION_MAX_ELECTRONIC_F1_DROP))
print()
for check_name, passed in ADOPTION_GATE_CHECKS.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
print()
print('--- interpretation aids (NOT gates) ---')
print('Gain exceeds 2x seed noise : {}  (|gain| {:.8f} vs 2*std {:.8f})'.format(
    GAIN_EXCEEDS_SEED_NOISE, abs(OOF_MACRO_F1_GAIN), 2.0 * CONTROL_SEED_SPREAD))
print('Capacity control gain      : {:+.10f}'.format(CAPACITY_GAIN))
print('Capacity explains >= half  : {}'.format(CAPACITY_EXPLAINS_GAIN))
if CAPACITY_EXPLAINS_GAIN:
    print('  -> A gram_texture win here is mostly extra parameters, NOT second-order structure.')
    print('     Do not describe it as evidence that texture covariance helps.')
if gram_oof['folds_hitting_epoch_cap'] >= 3:
    print('Epoch cap warning          : gram_texture hit the {}-epoch cap on {} of 5 folds, so it is'.format(
        MAX_EPOCHS, gram_oof['folds_hitting_epoch_cap']))
    print('     under-trained and this result is a LOWER BOUND, not a conclusion.')
print('Strong-evidence threshold  : {}  (>= {:+.4f})'.format(STRONG_EVIDENCE, STRONG_EVIDENCE_MACRO_F1_GAIN))
print()
print('=' * 78)
print('DECISION: {}'.format('ADOPT gram_texture' if GRAM_ADOPTED else 'REJECT gram_texture'))
print('SELECTED ARM FOR THE SUBMISSION: {}'.format(SELECTED_ARM))
print('REASON: {}'.format(SELECTION_REASON))
print('=' * 78)
print('selection_decision.json written BEFORE any test image is opened.')
print('Signature:', SELECTION_DECISION_SIGNATURE)

In [ ]:
def train_full_head(arm, first_order, gram, epochs):
    """Retrain one head from scratch on every official raw training row. No validation, no early stopping."""
    all_indices = np.arange(len(train_df), dtype=np.int64)
    class_counts, class_weights = class_weights_from(cached_labels.numpy())

    seed_everything(SEED)
    model = build_head(arm).to(HEAD_DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    # T_max stays at MAX_EPOCHS so the learning-rate trajectory matches the cross-validation runs.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    history = []
    started = time.perf_counter()

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        history.append({'stage': 'full_training', 'arm': arm, 'seed': SEED, 'epoch': epoch,
                        'train_loss': train_loss, 'learning_rate': learning_rate,
                        'epoch_seconds': epoch_seconds})
        print('  full_training {} epoch={:02d}/{:02d} loss={:.6f} lr={:.6e}'.format(
            arm, epoch, epochs, train_loss, learning_rate))
        return False

    run_training_epochs(model, first_order, gram, all_indices, epochs, SEED,
                        criterion, optimizer, scheduler, on_epoch_end)
    metadata = {
        'schema_version': 1, 'revision_marker': REVISION_MARKER, 'arm': arm,
        'trained_on': 'all {} official raw training rows'.format(len(train_df)),
        'epochs': epochs,
        'epoch_selection': 'median best epoch across the selected arm cross-validation folds',
        'early_stopping_used': False, 'warm_start_used': False, 'seed': SEED,
        'learning_rate': HEAD_LEARNING_RATE, 'weight_decay': HEAD_WEIGHT_DECAY,
        'batch_size': HEAD_BATCH_SIZE, 'label_smoothing': LABEL_SMOOTHING,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM, 'scheduler_t_max': MAX_EPOCHS,
        'train_class_counts': class_counts.tolist(),
        'parameter_count': sum(p.numel() for p in model.parameters()),
        'training_seconds': time.perf_counter() - started,
        'cache_signature': TRAIN_CACHE_MANIFEST['cache_signature'],
        'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    }
    return model, metadata, pd.DataFrame(history)


selected_best_epochs = [int(m['best_epoch']) for m in PER_FOLD_METRICS[SELECTED_ARM]]
FULL_TRAINING_EPOCHS = int(round(float(np.median(selected_best_epochs))))
if not 1 <= FULL_TRAINING_EPOCHS <= MAX_EPOCHS:
    raise RuntimeError('Derived full-training epoch count {} is out of range.'.format(FULL_TRAINING_EPOCHS))

print('Selected arm         :', SELECTED_ARM)
print('Per-fold best epochs :', selected_best_epochs)
print('Median -> full epochs:', FULL_TRAINING_EPOCHS, '(V10 used {})'.format(
    V10_REFERENCE['full_training_epochs']))
print()

FINAL_HEAD, FINAL_HEAD_METADATA, final_training_history = train_full_head(
    SELECTED_ARM, TRAIN_FIRST_ORDER, TRAIN_GRAM, FULL_TRAINING_EPOCHS)
FINAL_HEAD_PATH = ARTIFACT_DIR / 'final_head.pt'
atomic_torch_save(
    {'metadata': FINAL_HEAD_METADATA,
     'state_dict': {n: t.detach().cpu().clone() for n, t in FINAL_HEAD.state_dict().items()}},
    FINAL_HEAD_PATH,
)
write_csv_atomic(final_training_history, ARTIFACT_DIR / 'final_training_history.csv')
write_json_atomic(FINAL_HEAD_METADATA, ARTIFACT_DIR / 'final_head_metadata.json')
print()
print('Final head trained on all {} rows for {} epochs -> {}'.format(
    len(train_df), FULL_TRAINING_EPOCHS, FINAL_HEAD_PATH))

# ---------------------------------------------------------------------------
# TEST ISOLATION RELEASE
# ---------------------------------------------------------------------------
persisted_decision = read_json(SELECTION_DECISION_PATH) if SELECTION_DECISION_PATH.is_file() else {}
TEST_UNLOCK_CHECKS = {
    'selection_decision_file_exists': bool(persisted_decision),
    'selection_decision_signature_matches': (
        persisted_decision.get('selection_decision_signature') == SELECTION_DECISION_SIGNATURE),
    'train_cache_complete': bool(TRAIN_CACHE_MANIFEST.get('complete') is True
                                 and TRAIN_CACHE_MANIFEST['rows'] == EXPECTED_TRAIN_ROWS),
    'five_fold_oof_complete_for_every_arm': all(
        OOF_METRICS[arm]['rows'] == EXPECTED_TRAIN_ROWS for arm in ARM_ORDER),
    'final_head_exists': FINAL_HEAD_PATH.is_file(),
    # Read from the persisted decision, not live state, so rerunning this cell stays idempotent.
    'test_untouched_during_selection': (
        persisted_decision.get('selection_decision', {}).get('test_images_accessed') is False),
    'manual_test_labels_untouched': not TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
failed_unlock_checks = [name for name, passed in TEST_UNLOCK_CHECKS.items() if not passed]
if failed_unlock_checks:
    raise RuntimeError('Test isolation cannot be released. Failed prerequisites: {}.'.format(
        failed_unlock_checks))

TEST_ACCESS_STATE['test_images_accessed'] = True
print()
print('Test isolation released. All train-only prerequisites passed:')
for check_name in TEST_UNLOCK_CHECKS:
    print('  [PASS]', check_name)


def valid_test_dir(candidate):
    try:
        entries = list(candidate.iterdir())
    except OSError:
        return False
    return len(entries) == EXPECTED_TEST_ROWS and all(
        entry.is_file() and not entry.is_symlink() and entry.suffix.lower() in ALLOWED_IMAGE_SUFFIXES
        for entry in entries)


def resolve_test_dir():
    candidates = [Path.cwd() / 'data' / 'test', KAGGLE_WORKING / 'data' / 'test']
    candidates.extend(discover_named_dirs(KAGGLE_INPUT, 'test', max_depth=3))
    matches = {c.resolve() for c in candidates if c.is_dir() and valid_test_dir(c)}
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(p) for p in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError('Expected exactly one official test directory with {} images; found {}:'.format(
            EXPECTED_TEST_ROWS, len(matches)) + '\n' + rendered)
    return matches.pop()


def resolve_submission_template():
    matches = {}
    for root in (Path.cwd() / 'data', KAGGLE_WORKING / 'data', KAGGLE_INPUT):
        if not root.is_dir():
            continue
        for candidate in root.rglob('*.csv'):
            try:
                frame = pd.read_csv(candidate)
            except Exception:
                continue
            if list(frame.columns) == SUBMISSION_COLUMNS and len(frame) == EXPECTED_TEST_ROWS:
                matches[str(candidate.resolve())] = frame
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(p) for p in sorted(matches)) or '  (none)'
        raise RuntimeError('Expected exactly one official submission template; found {}:'.format(
            len(matches)) + '\n' + rendered)
    path = next(iter(matches))
    return Path(path), matches[path]


TEST_DIR = resolve_test_dir()
TEMPLATE_PATH, submission_template = resolve_submission_template()

stem_to_path = {}
for path in sorted(TEST_DIR.iterdir(), key=lambda item: item.name):
    if path.stem in stem_to_path:
        raise RuntimeError('Duplicate test image stem: {}'.format(path.stem))
    stem_to_path[path.stem] = path

template_ids = submission_template['id'].tolist()
if len(set(template_ids)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('The official template contains duplicate ids.')
ordered_test_paths = []
for raw_id in template_ids:
    key = Path(str(raw_id)).stem
    if key not in stem_to_path:
        raise RuntimeError('Template id {!r} has no matching image file in {}.'.format(raw_id, TEST_DIR))
    ordered_test_paths.append(str(stem_to_path[key].resolve()))
if len(set(ordered_test_paths)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('Template ids did not map to {} distinct test images.'.format(EXPECTED_TEST_ROWS))

test_df = pd.DataFrame({'image_id': [str(i) for i in template_ids], 'path': ordered_test_paths})
TEST_ORDER_SHA256 = sha256_json(test_df['image_id'].tolist())
print()
print('Official test directory  :', TEST_DIR)
print('Official template        :', TEMPLATE_PATH)
print('Template row-order sha256:', TEST_ORDER_SHA256)
print()

TEST_FIRST_ORDER, TEST_GRAM, TEST_CACHE_MANIFEST = build_feature_cache('test', test_df)
if tuple(TEST_FIRST_ORDER.shape) != (EXPECTED_TEST_ROWS,) + FIRST_ORDER_TAIL:
    raise RuntimeError('Test first-order tensor has shape {}.'.format(tuple(TEST_FIRST_ORDER.shape)))

train_identity = dict(TRAIN_CACHE_MANIFEST['cache_identity'])
test_identity = dict(TEST_CACHE_MANIFEST['cache_identity'])
row_specific_keys = {'split', 'rows', 'image_ids_sha256', 'labels_sha256', 'folds_sha256'}
shared_keys = (set(train_identity) & set(test_identity)) - row_specific_keys
if {k: train_identity[k] for k in shared_keys} != {k: test_identity[k] for k in shared_keys}:
    raise RuntimeError('Train and test caches were not produced under the same frozen-backbone contract.')
release_cuda_cache()

TEST_INFERENCE_STARTED_AT = time.perf_counter()
test_probabilities = predict_probabilities(
    FINAL_HEAD, TEST_FIRST_ORDER, TEST_GRAM, np.arange(len(test_df), dtype=np.int64))
TEST_INFERENCE_SECONDS = time.perf_counter() - TEST_INFERENCE_STARTED_AT
if test_probabilities.shape != (EXPECTED_TEST_ROWS, NUM_CLASSES) or not np.isfinite(test_probabilities).all():
    raise RuntimeError('Test probability matrix is incomplete or non-finite.')
write_npy_atomic(test_probabilities.astype(np.float32), ARTIFACT_DIR / 'test_probabilities.npy')

test_predictions = test_probabilities.argmax(axis=1).astype(np.int64)
submission = pd.DataFrame({'id': submission_template['id'].to_numpy(), 'predicted': test_predictions})
SUBMISSION_VALIDATION = {
    'row_count_is_expected': bool(len(submission) == EXPECTED_TEST_ROWS),
    'columns_are_exact': bool(list(submission.columns) == SUBMISSION_COLUMNS),
    'no_missing_values': bool(not submission.isna().any().any()),
    'ids_are_unique': bool(submission['id'].is_unique),
    'predictions_in_class_codes': bool(submission['predicted'].isin([0, 1, 2]).all()),
    'row_order_matches_template': bool(submission['id'].tolist() == submission_template['id'].tolist()),
}
failed_validation = [name for name, passed in SUBMISSION_VALIDATION.items() if not passed]
if failed_validation:
    raise RuntimeError('Submission validation failed: {}.'.format(failed_validation))

SUBMISSION_PATH = ARTIFACT_DIR / 'submission_{}.csv'.format(sanitize_for_filename(TEAM_NAME))
write_csv_atomic(submission, SUBMISSION_PATH)
PREDICTED_DISTRIBUTION = {str(c): int((test_predictions == c).sum()) for c in range(NUM_CLASSES)}

print()
print('Test inference: 1 forward pass, argmax over a single probability matrix.')
print('  no TTA, no hflip, no multi-crop, no fold averaging, no blending, no manual correction.')
print('Predicted class distribution :', PREDICTED_DISTRIBUTION)
print('V10 test distribution        :', V10_REFERENCE['test_predicted_class_distribution'])
print('  (diagnostic only; this comparison never influenced training, selection or the predictions.)')
print('Mean max probability         : {:.10f}'.format(float(test_probabilities.max(axis=1).mean())))
print('Submission written           :', SUBMISSION_PATH)
for check_name, passed in SUBMISSION_VALIDATION.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
if TEAM_NAME == 'CHANGE_ME':
    print()
    print('!' * 78)
    print('!! TEAM_NAME was never set, so the file is submission_CHANGE_ME.csv again.       !!')
    print('!! Rename it to submission_<YourTeamName>.csv before uploading.                  !!')
    print('!' * 78)

In [ ]:
# ---------------------------------------------------------------------------
# REPRODUCTION CHECK from the persisted checkpoint. V10 precedent: 2.98e-08.
# ---------------------------------------------------------------------------
reloaded_payload = torch.load(FINAL_HEAD_PATH, map_location='cpu', weights_only=False)
reloaded_head = build_head(SELECTED_ARM).to(HEAD_DEVICE)
reloaded_head.load_state_dict(reloaded_payload['state_dict'], strict=True)
reproduced_probabilities = predict_probabilities(
    reloaded_head, TEST_FIRST_ORDER, TEST_GRAM, np.arange(len(test_df), dtype=np.int64))
MAX_ABS_PROBABILITY_DIFFERENCE = float(np.abs(reproduced_probabilities - test_probabilities).max())
PREDICTIONS_IDENTICAL = bool(np.array_equal(reproduced_probabilities.argmax(axis=1), test_predictions))

CHECKPOINT_REPRODUCTION = {
    'checkpoint': str(FINAL_HEAD_PATH),
    'max_abs_probability_difference': MAX_ABS_PROBABILITY_DIFFERENCE,
    'tolerance': RELOAD_TOLERANCE,
    'within_tolerance': bool(MAX_ABS_PROBABILITY_DIFFERENCE <= RELOAD_TOLERANCE),
    'identical_argmax': PREDICTIONS_IDENTICAL,
    'v10_precedent_max_abs_probability_difference': 2.98e-08,
}
write_json_atomic(CHECKPOINT_REPRODUCTION, ARTIFACT_DIR / 'checkpoint_reproduction_check.json')
if not (CHECKPOINT_REPRODUCTION['within_tolerance'] and PREDICTIONS_IDENTICAL):
    raise RuntimeError(
        'Checkpoint reproduction failed: max abs probability difference {:.3e}, identical argmax {}. '
        'Do not submit this file.'.format(MAX_ABS_PROBABILITY_DIFFERENCE, PREDICTIONS_IDENTICAL))
del reloaded_head, reloaded_payload
release_cuda_cache()

# ---------------------------------------------------------------------------
# ERROR ANALYSIS ON THE SELECTED ARM (train OOF only, never test)
# ---------------------------------------------------------------------------
selected_oof = OOF_METRICS[SELECTED_ARM]
selected_pooled = OOF_PREDICTIONS[SELECTED_ARM]
wrong_rows = selected_pooled[selected_pooled['prediction'] != selected_pooled['label']].copy()
ro_rows = wrong_rows[((wrong_rows['label'] == 0) & (wrong_rows['prediction'] == 2))
                     | ((wrong_rows['label'] == 2) & (wrong_rows['prediction'] == 0))]
write_csv_atomic(wrong_rows.sort_values('confidence', ascending=False),
                 ARTIFACT_DIR / 'oof_errors_{}.csv'.format(SELECTED_ARM))
write_csv_atomic(ro_rows.sort_values('confidence', ascending=False),
                 ARTIFACT_DIR / 'oof_recyclable_organic_errors_{}.csv'.format(SELECTED_ARM))

# Which rows did the candidate fix, and which did it break, relative to the control?
control_pooled = OOF_PREDICTIONS[CONTROL_ARM]
gram_pooled = OOF_PREDICTIONS[GRAM_ARM]
control_correct = (control_pooled['prediction'] == control_pooled['label']).to_numpy()
gram_correct = (gram_pooled['prediction'] == gram_pooled['label']).to_numpy()
fixed_mask = (~control_correct) & gram_correct
broken_mask = control_correct & (~gram_correct)
paired_delta = pd.DataFrame({
    'image_id': control_pooled['image_id'],
    'label': control_pooled['label'],
    'control_prediction': control_pooled['prediction'],
    'gram_prediction': gram_pooled['prediction'],
    'control_confidence': control_pooled['confidence'],
    'gram_confidence': gram_pooled['confidence'],
    'outcome': np.where(fixed_mask, 'fixed_by_gram',
                        np.where(broken_mask, 'broken_by_gram', 'unchanged')),
})
write_csv_atomic(paired_delta[paired_delta['outcome'] != 'unchanged'],
                 ARTIFACT_DIR / 'paired_error_delta_control_vs_gram.csv')

ERROR_ANALYSIS = {
    'selected_arm': SELECTED_ARM,
    'oof_total_errors': selected_oof['total_errors'],
    'oof_ro_errors': selected_oof['ro_errors'],
    'recyclable_organic_share_of_errors': (
        selected_oof['ro_errors'] / selected_oof['total_errors'] if selected_oof['total_errors'] else 0.0),
    'oof_electronic_errors': selected_oof['electronic_errors'],
    'oof_high_confidence_errors': selected_oof['high_confidence_errors'],
    'rows_fixed_by_gram': int(fixed_mask.sum()),
    'rows_broken_by_gram': int(broken_mask.sum()),
    'net_rows_changed_by_gram': int(fixed_mask.sum() - broken_mask.sum()),
    'historical_v10_oof_total_errors': V10_REFERENCE['oof_total_errors'],
    'historical_v10_oof_ro_errors': V10_REFERENCE['oof_ro_errors'],
    'label_ceiling_note': (
        'Validation always uses the original official labels, so any true-accuracy improvement that '
        'crosses a mislabelled row LOWERS measured OOF. OOF therefore has a label-noise ceiling; see '
        'CONTEXT section 15.5. Read a small net change against that, not against a perfect-model ideal.'
    ),
    'test_images_inspected_for_model_decisions': False,
}
write_json_atomic(ERROR_ANALYSIS, ARTIFACT_DIR / 'error_analysis.json')

TEST_INFERENCE_MANIFEST = {
    'schema_version': 1, 'revision_marker': REVISION_MARKER,
    'selected_arm': SELECTED_ARM, 'selection_reason': SELECTION_REASON,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'decision_rule': 'argmax over the single probability matrix',
    'inference_passes': 1, 'inference_seconds': TEST_INFERENCE_SECONDS,
    'final_head_checkpoint': str(FINAL_HEAD_PATH), 'full_training_epochs': FULL_TRAINING_EPOCHS,
    'fold_or_seed_ensembling': False, 'probability_blending': False, 'test_time_augmentation': False,
    'horizontal_flip_inference': False, 'multi_crop_inference': False,
    'manual_prediction_correction': False, 'leaderboard_feedback_used': False,
    'previous_submissions_loaded': False, 'manual_test_labels_accessed': False,
    'predicted_class_distribution': PREDICTED_DISTRIBUTION,
    'mean_max_probability': float(test_probabilities.max(axis=1).mean()),
    'model_repository': TIMM_HF_REPO_ID, 'model_revision': RESOLVED_REVISION,
    'backbone_documented_for_final_report': MODEL_MANIFEST['backbone_documented_for_final_report'],
    'submission_path': str(SUBMISSION_PATH), 'submission_validation': SUBMISSION_VALIDATION,
    'template_path': str(TEMPLATE_PATH), 'test_dir': str(TEST_DIR),
    'test_order_sha256': TEST_ORDER_SHA256,
    'test_cache_signature': TEST_CACHE_MANIFEST['cache_signature'],
    'train_cache_signature': TRAIN_CACHE_MANIFEST['cache_signature'],
    'checkpoint_reproduction': CHECKPOINT_REPRODUCTION,
}
write_json_atomic(TEST_INFERENCE_MANIFEST, ARTIFACT_DIR / 'test_inference_manifest.json')

write_json_atomic({
    'schema_version': 1, 'revision_marker': REVISION_MARKER, 'experiment_name': EXPERIMENT_NAME,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'preflight_signature': PREFLIGHT_SIGNATURE,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256, 'labels_sha256': LABELS_SHA256,
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SHA256, 'test_order_sha256': TEST_ORDER_SHA256,
    'gram_projection_sha256': GRAM_PROJECTION_SHA256,
    'train_cache_signature': TRAIN_CACHE_MANIFEST['cache_signature'],
    'test_cache_signature': TEST_CACHE_MANIFEST['cache_signature'],
    'model_revision': RESOLVED_REVISION, 'environment': ENVIRONMENT_INFO, 'gpus': GPU_RUNTIME_INFO,
    'total_runtime_minutes': elapsed_minutes(),
}, ARTIFACT_DIR / 'reproducibility_manifest.json')

# ---------------------------------------------------------------------------
# THE ONE BLOCK TO COPY BACK
# ---------------------------------------------------------------------------
print()
print('#' * 78)
print('# V14 RESULT BLOCK  --  copy this whole block back')
print('#' * 78)
print('experiment                     : {}'.format(EXPERIMENT_NAME))
print('model revision                 : {} (matches V10: {})'.format(
    RESOLVED_REVISION, MODEL_REVISION_MATCHES_V10))
print('gram projection sha256         : {}'.format(GRAM_PROJECTION_SHA256))
print('gram carries signal (preflight): mean pairwise cosine {:.6f}, off-diag energy {:.4f}'.format(
    signal_stage['gram_mean_pairwise_cosine'], signal_stage['gram_off_diagonal_energy_share']))
print('pipeline integrity             : {}   delta {:+.10f}  (exact: {})'.format(
    'PASS' if PIPELINE_INTEGRITY_OK else 'FAIL', CONTROL_REPRODUCTION['macro_f1_delta'],
    CONTROL_REPRODUCTION['exactly_reproduced']))
for arm in ARM_ORDER:
    oof = OOF_METRICS[arm]
    role = 'DIAG' if arm in DIAGNOSTIC_ARMS else ('ctrl' if arm == CONTROL_ARM else 'CAND')
    print('---- {} [{}] params {:,} ----'.format(arm, role, ARM_PARAMETER_COUNTS[arm]))
    print('  OOF macro F1                 : {:.10f}'.format(oof['macro_f1']))
    print('  OOF total errors             : {}'.format(oof['total_errors']))
    print('  OOF R/O errors               : {}  (R->O {}, O->R {})'.format(
        oof['ro_errors'], oof['recyclable_to_organic'], oof['organic_to_recyclable']))
    print('  OOF Electronic F1            : {:.10f}  (errors {})'.format(
        oof['f1_electronic'], oof['electronic_errors']))
    print('  log loss                     : {:.10f}'.format(oof['log_loss']))
    print('  per fold macro F1            : {}'.format(
        {k: round(v, 10) for k, v in oof['per_fold_macro_f1'].items()}))
    print('  best epochs / folds at cap   : {} / {}'.format(oof['best_epochs'], oof['folds_hitting_epoch_cap']))
print('---- comparison ----------------------------------------------------------')
print('  OOF macro F1 gain            : {:+.10f}   (gate {:+.4f})'.format(
    OOF_MACRO_F1_GAIN, ADOPTION_MIN_OOF_MACRO_F1_GAIN))
print('  fold 0 gain                  : {:+.10f}'.format(DEVELOPMENT_FOLD_GAIN))
print('  assessment folds won         : {} of 4'.format(FOLDS_WON))
print('  Electronic F1 drop           : {:+.10f}'.format(ELECTRONIC_F1_DROP))
print('  gain by seed                 : {}'.format({k: round(v, 8) for k, v in SEED_GAINS.items()}))
print('  control seed std             : {:.8f}   (V13 measured {:.8f})'.format(
    CONTROL_SEED_SPREAD, V13_CONTROL_SEED_SPREAD))
print('  gain exceeds 2x seed noise   : {}'.format(GAIN_EXCEEDS_SEED_NOISE))
print('  capacity_control gain        : {:+.10f}  (explains >= half: {})'.format(
    CAPACITY_GAIN, CAPACITY_EXPLAINS_GAIN))
print('  rows fixed / broken by gram  : {} / {}  (net {})'.format(
    ERROR_ANALYSIS['rows_fixed_by_gram'], ERROR_ANALYSIS['rows_broken_by_gram'],
    ERROR_ANALYSIS['net_rows_changed_by_gram']))
print('---- grouped-validation diagnostic (never a gate) ------------------------')
if GROUPED_DIAGNOSTIC.get('succeeded'):
    print('  chosen tau / files grouped   : {:.2f} / {}  (V9 reference {} files)'.format(
        GROUPED_DIAGNOSTIC['chosen_threshold'], GROUPED_DIAGNOSTIC['files_in_multi_member_groups'],
        V9_REFERENCE_GROUPED_FILES))
    print('  grouped control macro F1     : {:.10f}'.format(
        GROUPED_DIAGNOSTIC['grouped_macro_f1'][CONTROL_ARM]))
    print('  grouped gram macro F1        : {:.10f}'.format(
        GROUPED_DIAGNOSTIC['grouped_macro_f1'][GRAM_ARM]))
    print('  grouped gain                 : {:+.10f}'.format(GROUPED_DIAGNOSTIC['grouped_gain']))
else:
    print('  FAILED (non-fatal): {}'.format(GROUPED_DIAGNOSTIC.get('error', 'unknown')))
print('---- gate ----------------------------------------------------------------')
for check_name, passed in ADOPTION_GATE_CHECKS.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
print('---- decision ------------------------------------------------------------')
print('  gram_texture adopted         : {}'.format(GRAM_ADOPTED))
print('  selected arm                 : {}'.format(SELECTED_ARM))
print('  full training epochs         : {}'.format(FULL_TRAINING_EPOCHS))
print('  test class distribution      : {}'.format(PREDICTED_DISTRIBUTION))
print('  V10 test class distribution  : {}'.format(V10_REFERENCE['test_predicted_class_distribution']))
print('  reproduction max abs diff    : {:.3e}  (identical argmax: {})'.format(
    MAX_ABS_PROBABILITY_DIFFERENCE, PREDICTIONS_IDENTICAL))
print('  submission                   : {}'.format(SUBMISSION_PATH))
print('  total runtime                : {:.0f} min'.format(elapsed_minutes()))
print('#' * 78)

## 9. Yang harus dibaca setelah notebook selesai

Sel terakhir mencetak **V14 RESULT BLOCK**. Salin blok itu apa adanya. Keputusan adopsi/tolak sudah dihitung otomatis oleh gerbang terkunci — tidak perlu ditafsir sendiri.

### 9.1 Urutan pemeriksaan

**Langkah 1 — integritas pipeline.** `pipeline integrity` harus `PASS`. V13 mencapai `delta 0.0000000000` persis; kalau kali ini delta ≠ 0 tapi masih di dalam toleransi, catat dan periksa apakah `model revision` berubah. Kalau `FAIL`, berhenti menafsirkan: seluruh angka di bawahnya tidak bisa dipercaya, adopsi otomatis ditolak, dan submission dibangun dari kontrol (= resep V10).

**Langkah 2 — apakah eksperimennya hidup.** `gram carries signal` dari preflight. V13 mati karena representasinya nyaris tidak bergerak (cosine 0,99941). Kalau angka Gram di sini juga ≥ 0,999, notebook sudah berhenti sebelum ekstraksi — kalau ia lanjut, berarti deskriptor orde-dua memang bervariasi antar-citra.

**Langkah 3 — gerbang.** Sembilan baris `PASS`/`FAIL`. Adopsi hanya kalau kesembilannya `PASS`.

**Langkah 4 — tiga penafsir (bukan gerbang), dan ketiganya bisa membatalkan klaim kemenangan:**

| Penafsir | Kalau begini, artinya |
|---|---|
| `gain exceeds 2x seed noise` = `False` | Gain lebih kecil dari derau seed. Jangan klaim menang walaupun gerbang lolos. V13 mengukur std = 0,000213, artinya gate +0,0003 hanya ≈ 1,4 σ. |
| `capacity_control gain` mendekati atau melebihi setengah gain kandidat | Yang menang adalah **parameter tambahan**, bukan struktur orde-dua. Tolak klaim mekanismenya. |
| `folds at cap` untuk `gram_texture` ≥ 3 | Kandidat *under-trained* pada batas 15 epoch. Hasilnya batas bawah, bukan kesimpulan — layak diulang dengan anggaran epoch lebih besar untuk kedua arm. |

**Langkah 5 — mekanisme.** Hipotesisnya spesifik: ko-okurensi tekstur menolong batas R/O. Jadi `OOF R/O errors` harus turun, dan `rows fixed / broken by gram` harus menunjukkan perbaikan bersih yang nyata. Kalau Macro F1 naik tanpa R/O turun, mekanismenya salah — dan gerbang memang sudah menolaknya lewat `oof_ro_errors_decrease`.

**Langkah 6 — grouped-CV (diagnostik).** Angka grouped hampir pasti **lebih rendah** dari angka stratified. Itu **koreksi pengukuran**, bukan regresi, dan bukan alasan menaikkan kompleksitas model. Yang penting: apakah tanda gain kontrol→kandidat tetap sama di kedua skema fold.

### 9.2 Empat kemungkinan hasil

| Hasil | Arti | Tindakan |
|---|---|---|
| Semua gerbang PASS, gain > 2× derau, R/O turun, kapasitas tidak menjelaskan | Champion baru, dan bukti bahwa readout orde-satu memang membuang informasi tekstur. | Submission dari `gram_texture` sudah dibuat & terverifikasi. Kandidat terkuat untuk submission terakhir. |
| Gerbang PASS tapi kapasitas menjelaskan sebagian besar gain | Bukan orde-dua yang menolong, cuma head lebih besar. | Jangan adopsi klaimnya. Kalau tetap ingin skor, `capacity_control` lebih jujur dan lebih sederhana — tapi itu perlu eksperimen sendiri. |
| Gain positif kecil, gerbang gagal | Orde-dua netral pada representasi ini. | Tolak. Submission dari kontrol. Cabang orde-dua ditutup. |
| Gain negatif | Cabang orde-dua mengganggu decision surface, sama seperti V8/V10-rich/V11. | Tolak, catat sebagai dead-end. Pola berulang: apa pun yang menambah kapasitas/jalur ke readout ini merugikan. |

### 9.3 Untuk submission terakhir — baca ini sebelum mengunggah

Tersisa **1 submission**. Tiga fakta yang harus dipegang bersamaan:

1. **Champion 0.9901662324 sebagian keberuntungan seed.** Kontrol yang sama menghasilkan 287 / 295 / 308 error pada seed 42 / 314 / 2026. Seed 42 kebetulan yang terbaik. Ekspektasi jujurnya ≈ 0,98990 ± 0,00021.
2. **OOF punya plafon label.** Validasi memakai label resmi asli, jadi perbaikan akurasi sebenarnya yang melewati sampel bermasalah justru **menurunkan** OOF (CONTEXT §15.5, dan V9 sudah membuktikannya secara empiris). Mengejar OOF 0,997 sama dengan menuntut model menghafal noise label.
3. **Skor leaderboard hampir linier terhadap jumlah error**: ≈ `100 − 0,0538 × k` pada 1.458 citra. Rank 20 = 7 error. Error rate OOF sekarang memproyeksikan ± 16 error.

Konsekuensinya: pilih model yang **stabil**, bukan yang kebetulan mencetak OOF tertinggi pada satu seed. Kalau `gram_texture` menang tipis dan tidak lolos `gain exceeds 2x seed noise`, **kontrol adalah pilihan yang lebih aman** untuk satu-satunya kesempatan tersisa — kandidat yang menang di dalam derau tidak punya alasan untuk menang di test.

Dan jangan lupa **isi `TEAM_NAME`**; run V13 keluar sebagai `submission_CHANGE_ME.csv`.

### 9.4 Yang sengaja tidak dikerjakan di sini

- **Varian kernel resize lain** (Lanczos, area, nearest): mati bersama V13. Cosine fitur bicubic vs bilinear 0,99941 — backbone-nya invarian terhadap kernel, bukan cuma tidak suka bicubic.
- **Cache embedding teman satu tim.** V14 butuh statistik orde-dua per citra yang tidak dimiliki cache mana pun yang ada, jadi ekstraksi ulang wajib apa pun protokolnya dan menumpang cache itu tidak menghemat apa pun. Kriteria audit untuk pemakaian lain tercatat di CONTEXT §15.7.
- **Arsitektur V11A1 (material-query attention).** Sudah ditolak di §8, dan gate adopsinya memakai *seed-averaged OOF* yang melanggar single-model contract. Yang diambil hanya ide desainnya: kontrol harus punya anggaran optimisasi yang sama — di V14 semua arm dilatih dari nol sehingga confound itu tidak bisa muncul.
- **V12 ViT-L + LoRA.** ViT-L bukan ViT-7B, jadi tidak boleh jadi kandidat submission akhir, dan 5 fold-nya tidak muat di sisa waktu (CONTEXT §14).
- **Label audit / relabel.** V9 sudah menguji drop dan relabel: keduanya **menurunkan** Macro F1 karena validasi tetap memakai label asli.

### 9.5 Artifact

```
/kaggle/working/v14_dinov3_vit7b_second_order_gram_texture/
├── submission_<TEAM_NAME>.csv           <- yang diunggah
├── final_head.pt                        <- checkpoint tunggal
├── selection_decision.json              <- ditulis SEBELUM test dibuka
├── control_reproduction.json            <- gerbang integritas pipeline
├── locked_contract.json                 <- gerbang dikunci sebelum ada hasil
├── preflight_report.json                <- termasuk identitas kecukupan Gram + uji "sinyal hidup"
├── model_manifest.json                  <- backbone untuk laporan akhir (juknis §5)
├── oof_metrics_<arm>.json, oof_predictions_<arm>.csv, oof_probabilities_<arm>.npy
├── per_fold_seed42.csv, seed_stability.csv
├── grouped_validation_diagnostic.json, near_duplicate_thresholds.csv
├── paired_error_delta_control_vs_gram.csv   <- baris mana yang diperbaiki / dirusak
├── oof_errors_<arm>.csv, oof_recyclable_organic_errors_<arm>.csv
├── test_probabilities.npy, test_inference_manifest.json
├── checkpoint_reproduction_check.json, reproducibility_manifest.json
└── feature_cache/{train,test}/           <- ± 4,3 GB, simpan sebagai Kaggle Dataset kalau sempat
```